In [1]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# import libraries
import pandas as pd
import numpy as np

# import fuzzy match
!pip install fuzzywuzzy
!pip install python-Levenshtein
from fuzzywuzzy import process


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 40.4 MB/s eta 0:00:00


In [3]:
# name file path
AI_Folder = '/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/Data sets & reports (O&G)/O&G global materials/Asset Impact Data/2025 Data'
US_file = AI_Folder + '/GEM-AI 2025 6 US states OG data.parquet'

In [4]:
# code from daan to open parquet
import pyarrow.parquet as pq

import os

# Folder that contains the Parquet file (fill this in)

location = AI_Folder
file_path = os.path.join(
    location,
    "GEM-AI 2025 6 US states OG data.parquet",

)

# Read the Parquet file
table = pq.read_table(file_path)
df = table.to_pandas()


In [5]:
# load in wyoming and ohio data as well
WyomingOhioSup = AI_Folder + '/GEM-AI 2025 Ohio Wyoming Update.parquet'
table = pq.read_table(WyomingOhioSup)
WyomingOhioSup_df = table.to_pandas()

# bring in new ohio and wyoming data
df = df[~df['state'].isin(['ohio', 'wyoming'])]
df = pd.concat([df, WyomingOhioSup_df], ignore_index=True)
print(f'DataFrame updated with Ohio and Wyoming data from WyomingOhioSup_df. New row count: {len(df)}')

DataFrame updated with Ohio and Wyoming data from WyomingOhioSup_df. New row count: 4736509


In [6]:
# check unique states included
unique_states_list = df['state'].unique()
unique_states_str = ', '.join(unique_states_list)
print('Unique states are: ')
print(unique_states_str)

# check unique columns included
unique_columns = df.columns
print('Unique columns are: ')
print(unique_columns)

Unique states are: 
texas, new_mexico, california, pennsylvania, ohio, wyoming
Unique columns are: 
Index(['country', 'state', 'county_name', 'year', 'well_no', 'well_name',
       'field_no', 'field_name', 'lease_no', 'lease_name', 'formation_no',
       'formation_name', 'longitude', 'latitude', 'operator_no',
       'operator_name', 'owner_no', 'owner_name', 'ownership_perc',
       'resource_type', 'resource_type_detail', 'recovery_mechanism',
       'field_terrain', 'oil_gravity', 'gas_gravity', 'oil_and_condensate_vol',
       'gross_gas_vol', 'dry_gas_vol', 'ngl_vol'],
      dtype='object')


In [7]:
# filter down to 2024 data
key_year = 2024
AIkeyyear = df[df['year'] == key_year]

In [8]:
# check data density (is not blank) per state for each attribute in given year (2024 here)
density_data = []
for state in unique_states_list:
  state_df = AIkeyyear[AIkeyyear['state'] == state]
  # Avoid division by zero for empty state_df
  if len(state_df) == 0:
    for column in unique_columns:
      density_data.append({"State": state, "Column": column, "Density": 0.0})
    continue

  for column in unique_columns:
    density = state_df[column].count() / len(state_df)
    density_data.append({"State": state, "Column": column, "Density": density})

import pandas as pd
density_df = pd.DataFrame(density_data)
display(density_df)

,State,Column,Density
0,texas,country,1.000000
1,texas,state,1.000000
2,texas,county_name,0.944437
3,texas,year,1.000000
4,texas,well_no,0.609354
...,...,...,...
169,wyoming,gas_gravity,0.000000
170,wyoming,oil_and_condensate_vol,1.000000
171,wyoming,gross_gas_vol,1.000000
172,wyoming,dry_gas_vol,1.000000


In [9]:
for state in unique_states_list:
  state_df = AIkeyyear[AIkeyyear['state'] == state]
  print(f'{state} has {len(state_df)} rows')
  # show state dfs names
  print(f'data frame for {state} has been created.')

texas has 155212 rows
data frame for texas has been created.
new_mexico has 65990 rows
data frame for new_mexico has been created.
california has 54978 rows
data frame for california has been created.
pennsylvania has 86110 rows
data frame for pennsylvania has been created.
ohio has 36484 rows
data frame for ohio has been created.
wyoming has 30561 rows
data frame for wyoming has been created.


As of last run 2025 01 21:
texas has 155212 rows
data frame for texas has been created.
new_mexico has 65990 rows
data frame for new_mexico has been created.
california has 54978 rows
data frame for california has been created.
pennsylvania has 86110 rows
data frame for pennsylvania has been created.
ohio has 36484 rows
data frame for ohio has been created.
wyoming has 30561 rows
data frame for wyoming has been created.

In [10]:
# calacuate the dry_gas_vol, gross_gas_vol, oil_and_condensate_vol, and ngl_vol for each states

premerge_totals_data = []

for state in unique_states_list:
  state_df = AIkeyyear[AIkeyyear['state'] == state]
  dry_gas_vol_sum = state_df['dry_gas_vol'].sum()
  gross_gas_vol_sum = state_df['gross_gas_vol'].sum()
  oil_and_condensate_vol_sum = state_df['oil_and_condensate_vol'].sum()
  ngl_vol_sum = state_df['ngl_vol'].sum()

  premerge_totals_data.append({
      'State': state,
      'Dry Gas Volume Total': dry_gas_vol_sum,
      'Gross Gas Volume Total': gross_gas_vol_sum,
      'Oil and Condensate Volume Total': oil_and_condensate_vol_sum,
      'NGL Volume Total': ngl_vol_sum
  })

premerge_totals_df = pd.DataFrame(premerge_totals_data)
display(premerge_totals_df)

,State,Dry Gas Volume Total,Gross Gas Volume Total,Oil and Condensate Volume Total,NGL Volume Total
0,texas,288552.264139,374899.221792,2062.997260,824.270919
1,new_mexico,91954.413333,101316.953696,739.031557,212.000645
2,california,3210.686283,3414.208182,112.597974,4.452663
3,pennsylvania,207583.814717,211087.172699,4.460972,85.211735
4,ohio,58949.378280,61010.740399,36.670902,50.970204
5,wyoming,27803.795099,38059.534215,106.747987,43.479693


Gas volumes are listed in million cubic meters/y, liquids in million bbl/y

In [11]:
# save out density_df
density_df.to_csv(f'{AI_Folder}/density_{key_year}.csv', index=False)
print(f'saved density_{key_year}.csv')

saved density_2024.csv


In [13]:
# check texas offshore
Texas = AIkeyyear[AIkeyyear['state'] == 'texas']
Texas_offshore = Texas[Texas['field_terrain'] == 'Offshore']

# save to output
Texas_offshore.to_csv(f'{AI_Folder}/texas_offshore_{key_year}.csv', index=False)

# Wyoming
Wyoming was not updated in the 2025 GOGET release. Data is currently at the Field Name + County + Operator (n=205)

In [12]:
# Install fuzzywuzzy if not already installed
!pip install fuzzywuzzy
!pip install python-Levenshtein

from fuzzywuzzy import process
import numpy as np

In [ ]:
# group wyoming df by field_name + operator_name
wyoming_df = AIkeyyear[AIkeyyear['state'] == 'wyoming']

# title case operator and field name
wyoming_df_capitalized = wyoming_df.copy()
wyoming_df_capitalized['operator_name'] = wyoming_df['operator_name'].str.title()
wyoming_df_capitalized['field_name'] = wyoming_df['field_name'].str.title()
wyoming_df_capitalized['formation_name'] = wyoming_df['formation_name'].str.title()

# Convert longitude and latitude to numeric, coercing errors to NaN
wyoming_df_capitalized['longitude'] = pd.to_numeric(wyoming_df_capitalized['longitude'], errors='coerce')
wyoming_df_capitalized['latitude'] = pd.to_numeric(wyoming_df_capitalized['latitude'], errors='coerce')

# fill in "formation not assigned" when formation_name = null
wyoming_df_capitalized['formation_name'] = wyoming_df_capitalized['formation_name'].fillna('Formation not assigned')
wyoming_df_capitalized['field_name'] = wyoming_df_capitalized['field_name'].fillna('Field not assigned')

# Replace "wc" in field_name with "Wild cat"
wyoming_df_capitalized['field_name'] = wyoming_df_capitalized['field_name'].str.replace('WC', 'Wild cat')

print("data cleaned")

data cleaned


In [ ]:
# group by operator + field
wyoming_df_grouped_field = wyoming_df_capitalized.groupby(['field_name', 'operator_name']).agg({
    'well_no': lambda x: '; '.join(x.astype(str).unique()),
    'longitude': 'mean',
    'latitude': 'mean',
    'gross_gas_vol': 'sum',
    'dry_gas_vol': 'sum',
    'oil_and_condensate_vol': 'sum',
    'ngl_vol': 'sum',
    'formation_name': lambda x: '; '.join(x.astype(str).unique()), # Include formation_name in aggregation
    'resource_type': lambda x: '; '.join(x.astype(str).unique()) # Include resource_type in aggregation
}).reset_index()

# count grouped rows
print(f'Wyoming has {len(wyoming_df_grouped_field)} rows when grouped by field + operator')
# print count of first not assinged entries
print(f'Wyoming has {len(wyoming_df_grouped_field[wyoming_df_grouped_field["field_name"] == "Field not assigned"])} rows with "Field not assigned"')

Wyoming has 1381 rows when grouped by field + operator
Wyoming has 1 rows with "Field not assigned"


In [ ]:
# group wyoming df by field_name + operator_name
wyoming_df_grouped_formation = wyoming_df_capitalized.groupby(['formation_name', 'operator_name']).agg({
    'well_no': lambda x: '; '.join(x.astype(str).unique()),
    'longitude': 'mean',
    'latitude': 'mean',
    'gross_gas_vol': 'sum',
    'dry_gas_vol': 'sum',
    'oil_and_condensate_vol': 'sum',
    'ngl_vol': 'sum',
    'resource_type': lambda x: '; '.join(x.astype(str).unique()) # Include resource_type in aggregation
}).reset_index()

# count grouped rows
print(f'Wyoming has {len(wyoming_df_grouped_formation)} rows when grouped by formation + operator')

# print count of "Formation not assigned" entries
print(f'Wyoming has {len(wyoming_df_grouped_formation[wyoming_df_grouped_formation["formation_name"] == "Formation not assigned"])} rows with "Formation not assigned"')

Wyoming has 1195 rows when grouped by formation + operator
Wyoming has 9 rows with "Formation not assigned"


In [ ]:
# check totals in formation grouping
WY_dry_gas_vol_sum = wyoming_df_grouped_formation['dry_gas_vol'].sum()
WY_gross_gas_vol_sum = wyoming_df_grouped_formation['gross_gas_vol'].sum()
WY_oil_and_condensate_vol_sum = wyoming_df_grouped_formation['oil_and_condensate_vol'].sum()
WY_ngl_vol_sum = wyoming_df_grouped_formation['ngl_vol'].sum()

print(f'formation grouped dry gas sum = {WY_dry_gas_vol_sum}')
print(f'formation grouped gross gas sum = {WY_gross_gas_vol_sum}')
print(f'formation grouped oil and condensate sum = {WY_oil_and_condensate_vol_sum}')
print(f'formation grouped ngl sum = {WY_ngl_vol_sum}')

formation grouped dry gas sum = 27803.79509938584
formation grouped gross gas sum = 38059.53421505389
formation grouped oil and condensate sum = 106.747987
formation grouped ngl sum = 43.47969269745806


In [ ]:
# check totals in field grouping
WY_dry_gas_vol_sum = wyoming_df_grouped_field['dry_gas_vol'].sum()
WY_gross_gas_vol_sum = wyoming_df_grouped_field['gross_gas_vol'].sum()
WY_oil_and_condensate_vol_sum = wyoming_df_grouped_field['oil_and_condensate_vol'].sum()
WY_ngl_vol_sum = wyoming_df_grouped_field['ngl_vol'].sum()

print(f'field grouped dry gas sum= {WY_dry_gas_vol_sum}')
print(f'field grouped gross gas sum = {WY_gross_gas_vol_sum}')
print(f'field grouped oil and condensate sum = {WY_oil_and_condensate_vol_sum}')
print(f'field grouped ngl sum = {WY_ngl_vol_sum}')



field grouped dry gas sum= 27803.79509938584
field grouped gross gas sum = 38059.53421505389
field grouped oil and condensate sum = 106.747987
field grouped ngl sum = 43.47969269745806


In [ ]:
# pull in GEM Entity IDs for top 95% operators
WY_top_op = pd.read_csv(f'{AI_Folder}/wyoming_grouped_formation_2024 - Unique Operators.csv')

# pull in field discovery years from GOGET
WY_GOGET_Folder = '/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/Data sets & reports (O&G)/O&G country materials/United States (GOGET O&G)/Wyoming (GOGET)/'
WY_disco_years = pd.read_csv(WY_GOGET_Folder + 'GOGET WY 2025 12 12 - GOGET fields discovery year.csv')

In [ ]:
# merge Entity ID based on Operator	in WY_top_op and operator_name in wyoming_df_grouped_formation
wyoming_df_grouped_formation_plusID = wyoming_df_grouped_formation.merge(WY_top_op, left_on='operator_name', right_on='Operator', how='left')
wyoming_df_grouped_field_plusID = wyoming_df_grouped_field.merge(WY_top_op, left_on='operator_name', right_on='Operator', how='left')

# drop columns from wyoming_df_grouped_formation
WY_Form_ID = wyoming_df_grouped_formation_plusID.drop(columns=['Operator', 'Production','Percent here','Percentage of Wyoming Gas Production by Company (2023), per PAW', 'Percentage of Wyoming Oil Production by Company (2023)'])
WY_Field_ID = wyoming_df_grouped_field_plusID.drop(columns=['Operator', 'Production','Percent here','Percentage of Wyoming Gas Production by Company (2023), per PAW', 'Percentage of Wyoming Oil Production by Company (2023)'])

In [ ]:
# drop rows where Entity ID is null
WY_Form_ID_noNull = WY_Form_ID.dropna(subset=['Entity ID']).copy()
WY_Field_ID_noNull = WY_Field_ID.dropna(subset=['Entity ID']).copy()

# print row count pre and post drop formation and field
print(f'Formation grouped has {len(wyoming_df_grouped_formation)} rows')
print(f'Formation groups with only top ops has {len(WY_Form_ID_noNull)} rows')
print(f'Field grouped has {len(wyoming_df_grouped_field)} rows')
print(f'Field groups with only top ops has {len(WY_Field_ID_noNull)} rows')

Formation grouped has 1195 rows
Formation groups with only top ops has 413 rows
Field grouped has 1381 rows
Field groups with only top ops has 434 rows


In [ ]:
# calcuate sums for gross_gas, dry gas, oil and cond, NGL in new dataframe for formations
filter_WY_dry_gas_vol_sum = WY_Form_ID_noNull['dry_gas_vol'].sum()
filter_WY_gross_gas_vol_sum = WY_Form_ID_noNull['gross_gas_vol'].sum()
filter_WY_oil_and_condensate_vol_sum = WY_Form_ID_noNull['oil_and_condensate_vol'].sum()
filter_WY_ngl_vol_sum = WY_Form_ID_noNull['ngl_vol'].sum()

# print sums
print(f'WY_Form_ID_noNull_sum dry_gas_vol = {filter_WY_dry_gas_vol_sum}')
print(f'WY_Form_ID_noNull_sum gross_gas_vol = {filter_WY_gross_gas_vol_sum}')
print(f'WY_Form_ID_noNull_sum oil_and_condensate_vol = {filter_WY_oil_and_condensate_vol_sum}')
print(f'WY_Form_ID_noNull_sum ngl_vol = {filter_WY_ngl_vol_sum}')

WY_Form_ID_noNull_sum dry_gas_vol = 27165.000385795287
WY_Form_ID_noNull_sum gross_gas_vol = 37185.112965314795
WY_Form_ID_noNull_sum oil_and_condensate_vol = 98.50012399999999
WY_Form_ID_noNull_sum ngl_vol = 42.480742815098615


In [ ]:
# calcuate sums for gross_gas, dry gas, oil and cond, NGL in new dataframe for fields
filter_WY_dry_gas_vol_sum = WY_Field_ID_noNull['dry_gas_vol'].sum()
filter_WY_gross_gas_vol_sum = WY_Field_ID_noNull['gross_gas_vol'].sum()
filter_WY_oil_and_condensate_vol_sum = WY_Field_ID_noNull['oil_and_condensate_vol'].sum()
filter_WY_ngl_vol_sum = WY_Field_ID_noNull['ngl_vol'].sum()

# print sums
print(f'WY_Field_ID_noNull dry_gas_vol = {filter_WY_dry_gas_vol_sum}')
print(f'WY_Field_ID_noNull gross_gas_vol = {filter_WY_gross_gas_vol_sum}')
print(f'WY_Field_ID_noNull oil_and_condensate_vol = {filter_WY_oil_and_condensate_vol_sum}')
print(f'WY_Field_ID_noNull ngl_vol = {filter_WY_ngl_vol_sum}')

WY_Field_ID_noNull dry_gas_vol = 27165.000385795287
WY_Field_ID_noNull gross_gas_vol = 37185.112965314795
WY_Field_ID_noNull oil_and_condensate_vol = 98.500124
WY_Field_ID_noNull ngl_vol = 42.480742815098615


In [ ]:
# set row for choosing field or formation
row_choice = 'field' # User can change select 'formation' and 'field'

if row_choice == 'formation':
    df_to_use = WY_Form_ID_noNull
    name_col = 'formation_name'
    project_complex_col = 'formation_name'
elif row_choice == 'field':
    df_to_use = WY_Field_ID_noNull
    name_col = 'field_name'
    project_complex_col = 'formation_name'
else:
    raise ValueError("Invalid row_choice. Must be 'formation' or 'field'.")

choice = df_to_use

print(f'choice is {row_choice}')
choice.head()

choice is field


,field_name,operator_name,well_no,longitude,latitude,gross_gas_vol,dry_gas_vol,oil_and_condensate_vol,ngl_vol,formation_name,resource_type,Entity ID
2,African Swallow,"Sage Butte Prb Operating, Llc",929319; 929768,-105.862505,43.359510,2.062656,1.506841,0.011325,0.002356,Frontier,Unconventional; Conventional,E100002021104
3,African Swallow,Wrc Energy Llc,922959; 922973,NaN,NaN,5.055463,3.693189,0.004545,0.005775,Muddy,Conventional,E100002021101
4,Alicia,Wrc Energy Llc,527613; 528381; 528394; 528482,-105.886890,44.390950,0.953796,0.696781,0.001030,0.001090,Muddy,Conventional,E100002021101
12,Amos Draw,Wrc Energy Llc,527047; 527190; 527202; 527218; 527219; 527230...,-105.904745,44.373727,1.831279,1.337812,0.001488,0.002092,Muddy,Conventional,E100002021101
15,Andy,Wrc Energy Llc,526958; 526970; 527043; 527376; 527385; 527797...,-105.953740,44.349624,2.136619,1.560874,0.001973,0.002441,Muddy,Conventional,E100002021101


In [ ]:
# Map resource type : Unconventional; Conventional, Conventional; Unconventional and Unconventional = "Unconventional". Conventional = Conventional
choice['resource_type'] = choice['resource_type'].replace({
    'Unconventional; Conventional': 'Unconventional',
    'Conventional; Unconventional': 'Unconventional',
    'Conventional': 'Conventional'
})

In [ ]:
import numpy as np

conditions = [
    (choice['dry_gas_vol'] > 0) & (choice['oil_and_condensate_vol'] > 0),
    (choice['dry_gas_vol'] == 0) & (choice['oil_and_condensate_vol'] > 0),
    (choice['dry_gas_vol'] > 0) & (choice['oil_and_condensate_vol'] == 0)
]

choices = ['oil and gas', 'oil', 'gas']

choice['fuel_type'] = np.select(conditions, choices, default=None)

In [ ]:
# merge discovery year on to choice based on field name
choice = choice.merge(WY_disco_years, left_on='field_name', right_on='Unit Name', how='left')

# drop columns from wyoming_df_grouped_formation
choice = choice.drop(columns=['Unit Name'])

In [ ]:
# format for GOGET bulk upload
# create asset name
choice.loc[:, 'name'] = choice[name_col] + ' - ' + choice['operator_name']

# add unit type
choice.loc[:, 'unit_type'] = "asset"

# add project/complex
choice.loc[:, 'project_complex'] = choice[project_complex_col]

# add coordinate accuracy
choice.loc[:, 'coordinate_accuracy'] = "exact"

# add operator source
choice.loc[:, 'operator_source'] = "WOGCC_Op_2025"

# add location source
choice.loc[:, 'location_source'] = "WOGCC_Loc_2025; WOGCC_Prod_2024"

# add resource source
choice.loc[:, 'resource_type_source'] = "WOGCC_Prod_2024"

# add discovery year source
choice.loc[:, 'discovery_year_source'] = "WOGCC_Prod_2024"

# replace "Wc" in name with "Wildcat"
choice['name'] = choice['name'].str.replace('Wc', 'Wildcat')

In [ ]:
choice.columns

Index(['field_name', 'operator_name', 'well_no', 'longitude', 'latitude',
       'gross_gas_vol', 'dry_gas_vol', 'oil_and_condensate_vol', 'ngl_vol',
       'formation_name', 'resource_type', 'Entity ID', 'fuel_type',
       'Discovery year', 'name', 'unit_type', 'project_complex',
       'coordinate_accuracy', 'operator_source', 'location_source',
       'resource_type_source', 'discovery_year_source'],
      dtype='object')

In [ ]:
print(f'Number of rows in choice before deduplication: {len(choice)}')
choice.drop_duplicates(inplace=True)
print(f'Number of rows in choice after deduplication: {len(choice)}')

Number of rows in choice before deduplication: 593
Number of rows in choice after deduplication: 434


In [ ]:
# add total production column
choice['total_hydrocarbons_production'] = (choice['dry_gas_vol']*0.005883) + choice['oil_and_condensate_vol'] + choice['ngl_vol']

# show sum of total hydrocarbon
total_hydrocarbons_production = choice['total_hydrocarbons_production'].sum()
print(f'total hydrocarbon production = {total_hydrocarbons_production}')

total hydrocarbon production = 300.7925640847323


In [ ]:
# set column order
# if choice = field, columns are 'name', 'unit_type', 'fuel_type', 'project_complex', 'longitude', 'latitude','coordinate_accuracy', 'location_source', 'operator_name', 'operator_source', 'Entity ID', 'resource_type', 'resource_type_source','Discovery year','discovery_year_source','well_no', 'total_hydrocarbons_production'
if row_choice == 'field': WY_Clean = choice[['name', 'unit_type', 'fuel_type', 'project_complex', 'longitude', 'latitude','coordinate_accuracy', 'location_source', 'operator_name', 'operator_source', 'Entity ID', 'resource_type', 'resource_type_source','Discovery year','discovery_year_source','well_no', 'total_hydrocarbons_production']].copy()
elif row_choice == 'formation': WY_Clean = choice[['name', 'unit_type', 'fuel_type', 'project_complex', 'longitude', 'latitude','coordinate_accuracy', 'location_source', 'operator_name', 'operator_source', 'Entity ID', 'resource_type', 'resource_type_source','well_no', 'total_hydrocarbons_production']].copy()

In [ ]:
# import existing GOGET IDs and names
GOGET_IDs = pd.read_csv(f'{WY_GOGET_Folder}Existing GOGET IDs.csv')

In [ ]:
# Create a list of Unit Names from GOGET_IDs for fuzzy matching -
goget_unit_names = GOGET_IDs['Unit Name'].tolist()

# Function to perform fuzzy matching and return the Unit ID
def fuzzy_match_unit_id(row, choices, ids_df, threshold=88): # Increased threshold to 90
    match = process.extractOne(row['name'], choices, score_cutoff=threshold)
    if match:
        # match[0] is the matched 'Unit Name', match[1] is the score
        # We need to find the original index in GOGET_IDs to get the Unit ID
        original_unit_name = match[0]
        unit_id = ids_df[ids_df['Unit Name'] == original_unit_name]['Unit ID'].iloc[0]
        return unit_id
    return np.nan

# Apply the fuzzy matching function to WY_Clean
WY_Clean['GOGET_Unit_ID'] = WY_Clean.apply(fuzzy_match_unit_id, axis=1, choices=goget_unit_names, ids_df=GOGET_IDs)

# calculate how many matches were made
matches = len(WY_Clean[WY_Clean['GOGET_Unit_ID'].notna()])

print("Fuzzy matching complete.")
print(f"{matches}" +' matches were made')

print("Displaying first 5 rows with new GOGET_Unit_ID:")
display(WY_Clean.head())

Fuzzy matching complete.
94 matches were made
Displaying first 5 rows with new GOGET_Unit_ID:


,name,unit_type,fuel_type,project_complex,longitude,latitude,coordinate_accuracy,location_source,operator_name,operator_source,Entity ID,resource_type,resource_type_source,Discovery year,discovery_year_source,well_no,total_hydrocarbons_production,GOGET_Unit_ID
0,"African Swallow - Sage Butte Prb Operating, Llc",asset,oil and gas,Frontier,-105.862505,43.359510,exact,WOGCC_Loc_2025; WOGCC_Prod_2024,"Sage Butte Prb Operating, Llc",WOGCC_Op_2025,E100002021104,Unconventional,WOGCC_Prod_2024,NaN,WOGCC_Prod_2024,929319; 929768,0.022546,NaN
1,African Swallow - Wrc Energy Llc,asset,oil and gas,Muddy,NaN,NaN,exact,WOGCC_Loc_2025; WOGCC_Prod_2024,Wrc Energy Llc,WOGCC_Op_2025,E100002021101,Conventional,WOGCC_Prod_2024,NaN,WOGCC_Prod_2024,922959; 922973,0.032047,NaN
2,Alicia - Wrc Energy Llc,asset,oil and gas,Muddy,-105.886890,44.390950,exact,WOGCC_Loc_2025; WOGCC_Prod_2024,Wrc Energy Llc,WOGCC_Op_2025,E100002021101,Conventional,WOGCC_Prod_2024,NaN,WOGCC_Prod_2024,527613; 528381; 528394; 528482,0.006219,NaN
3,Amos Draw - Wrc Energy Llc,asset,oil and gas,Muddy,-105.904745,44.373727,exact,WOGCC_Loc_2025; WOGCC_Prod_2024,Wrc Energy Llc,WOGCC_Op_2025,E100002021101,Conventional,WOGCC_Prod_2024,NaN,WOGCC_Prod_2024,527047; 527190; 527202; 527218; 527219; 527230...,0.011450,NaN
4,Andy - Wrc Energy Llc,asset,oil and gas,Muddy,-105.953740,44.349624,exact,WOGCC_Loc_2025; WOGCC_Prod_2024,Wrc Energy Llc,WOGCC_Op_2025,E100002021101,Conventional,WOGCC_Prod_2024,NaN,WOGCC_Prod_2024,526958; 526970; 527043; 527376; 527385; 527797...,0.013597,NaN


In [ ]:
production_cols = ['gross_gas_vol', 'dry_gas_vol', 'oil_and_condensate_vol', 'ngl_vol']
id_cols = [col for col in choice.columns if col not in production_cols] # This gets all non-production cols from 'choice'

WY_long_prod = pd.melt(choice,
                                  id_vars=id_cols,
                                  value_vars=production_cols,
                                  var_name='production_type',
                                  value_name='volume')

# Define columns to drop, based on which ones are present in the melted DataFrame
cols_to_drop_from_melted = ['Entity ID','well_no','longitude','latitude','unit_type', 'project_complex','coordinate_accuracy','operator_source','location_source', 'field_name','resource_type',	'resource_type_source',	'fuel_type',	'discovery_year_source',	'Discovery year']
if 'formation_name' in choice.columns:
    cols_to_drop_from_melted.append('formation_name')
elif 'field_name' in choice.columns:
    cols_to_drop_from_melted.append('field_name')
if 'operator_name' in choice.columns:
    cols_to_drop_from_melted.append('operator_name')

# Filter the list to only include columns actually present in WY_long_prod
existing_cols_to_drop = [col for col in cols_to_drop_from_melted if col in WY_long_prod.columns]
WY_long_prod.drop(columns=existing_cols_to_drop, inplace=True)

In [ ]:
# replace production type entries as follows, 'gross_gas_vol' = gross gas 'dry_gas_vol' = dry gas, 'oil_and_condensate_vol' = oil and condensates, 'ngl_vol' = NGL
WY_long_prod['fuel description'] = WY_long_prod['production_type'].replace({'gross_gas_vol': 'gross gas', 'dry_gas_vol': 'dry gas', 'oil_and_condensate_vol': 'oil and condensates', 'ngl_vol': 'NGL'})

# add production year column, add entries as 2024
WY_long_prod['Year'] = 2024

# add "Production/Reserves" column, set all rows as "production"
WY_long_prod['production/reserves'] = 'production'

# rename volume as "Quantity"
WY_long_prod.rename(columns={'volume': 'Quantity'}, inplace=True)

# add quantity unit column, if gross gas or dry gas, set as million cubic meters/y
WY_long_prod['quantity_unit'] = WY_long_prod['fuel description'].apply(lambda x: 'million cubic meters/y' if x in ['gross gas', 'dry gas'] else 'million bbl/y')

# add it production source
WY_long_prod['Datasource'] = 'WOGCC_Prod_2024'


In [ ]:
print(f'Number of rows in WY_long_prod before deduplication: {len(WY_long_prod)}')
WY_long_prod.drop_duplicates(inplace=True)
print(f'Number of rows in WY_long_prod after deduplication: {len(WY_long_prod)}')

Number of rows in WY_long_prod before deduplication: 1736
Number of rows in WY_long_prod after deduplication: 1736


In [ ]:
# sum entries by production_type in WY_long_prod
WY_long_prod_gas = WY_long_prod[WY_long_prod['fuel description'] == 'dry gas']['Quantity'].sum()
WY_long_prod_oil = WY_long_prod[WY_long_prod['fuel description'] == 'oil and condensates']['Quantity'].sum()

print(f"Total sales gas volume: {WY_long_prod_gas}")
print(f"Total oil and condensates volume: {WY_long_prod_oil}")

Total sales gas volume: 27165.000385795287
Total oil and condensates volume: 98.500124


In [ ]:
WY_Clean

,name,unit_type,fuel_type,project_complex,longitude,latitude,coordinate_accuracy,location_source,operator_name,operator_source,Entity ID,resource_type,resource_type_source,Discovery year,discovery_year_source,well_no,total_hydrocarbons_production,GOGET_Unit_ID
0,"African Swallow - Sage Butte Prb Operating, Llc",asset,oil and gas,Frontier,-105.862505,43.359510,exact,WOGCC_Loc_2025; WOGCC_Prod_2024,"Sage Butte Prb Operating, Llc",WOGCC_Op_2025,E100002021104,Unconventional,WOGCC_Prod_2024,NaN,WOGCC_Prod_2024,929319; 929768,0.022546,NaN
1,African Swallow - Wrc Energy Llc,asset,oil and gas,Muddy,NaN,NaN,exact,WOGCC_Loc_2025; WOGCC_Prod_2024,Wrc Energy Llc,WOGCC_Op_2025,E100002021101,Conventional,WOGCC_Prod_2024,NaN,WOGCC_Prod_2024,922959; 922973,0.032047,NaN
2,Alicia - Wrc Energy Llc,asset,oil and gas,Muddy,-105.886890,44.390950,exact,WOGCC_Loc_2025; WOGCC_Prod_2024,Wrc Energy Llc,WOGCC_Op_2025,E100002021101,Conventional,WOGCC_Prod_2024,NaN,WOGCC_Prod_2024,527613; 528381; 528394; 528482,0.006219,NaN
3,Amos Draw - Wrc Energy Llc,asset,oil and gas,Muddy,-105.904745,44.373727,exact,WOGCC_Loc_2025; WOGCC_Prod_2024,Wrc Energy Llc,WOGCC_Op_2025,E100002021101,Conventional,WOGCC_Prod_2024,NaN,WOGCC_Prod_2024,527047; 527190; 527202; 527218; 527219; 527230...,0.011450,NaN
4,Andy - Wrc Energy Llc,asset,oil and gas,Muddy,-105.953740,44.349624,exact,WOGCC_Loc_2025; WOGCC_Prod_2024,Wrc Energy Llc,WOGCC_Op_2025,E100002021101,Conventional,WOGCC_Prod_2024,NaN,WOGCC_Prod_2024,526958; 526970; 527043; 527376; 527385; 527797...,0.013597,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
588,Windmill Draw - Rangeview Resources Operating Co,asset,gas,Almond,-107.928830,41.284213,exact,WOGCC_Loc_2025; WOGCC_Prod_2024,Rangeview Resources Operating Co,WOGCC_Op_2025,E100000191561,Conventional,WOGCC_Prod_2024,NaN,WOGCC_Prod_2024,3721071; 3721785; 3722185,0.003710,NaN
589,Wood - M & K Oil Company Llc,asset,oil,Lakota,-104.977415,44.400825,exact,WOGCC_Loc_2025; WOGCC_Prod_2024,M & K Oil Company Llc,WOGCC_Op_2025,E100000191410,Conventional,WOGCC_Prod_2024,NaN,WOGCC_Prod_2024,1105722; 1105728,0.000494,NaN
590,Woodruff Narrows - Hilcorp Energy Company,asset,gas,Big Horn,-111.005500,41.497990,exact,WOGCC_Loc_2025; WOGCC_Prod_2024,Hilcorp Energy Company,WOGCC_Op_2025,E100001015870,Conventional,WOGCC_Prod_2024,NaN,WOGCC_Prod_2024,4120627,0.022299,NaN
591,York North - True Oil Llc,asset,oil,Minnelusa,-105.170560,44.621115,exact,WOGCC_Loc_2025; WOGCC_Prod_2024,True Oil Llc,WOGCC_Op_2025,E100000192296,Conventional,WOGCC_Prod_2024,NaN,WOGCC_Prod_2024,526437; 570356,0.001887,NaN


In [ ]:
# create time stamp
import datetime

time_stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

print(time_stamp)

2025-12-22_20-17-41


In [ ]:
# save as csv

# wyoming_df_grouped_formation.to_csv(f'{WY_GOGET_Folder}/wyoming_grouped_formation_{key_year}.csv', index=False)
WY_Clean.to_csv(f'{WY_GOGET_Folder}/WY_AI_FOR_GOGET{key_year}_{time_stamp}.csv', index=False)
WY_long_prod.to_csv(f'{WY_GOGET_Folder}/wyoming_grouped_formation_long_{key_year}.csv', index=False)

# Ohio Gemini Task


## Filter Ohio Data

### Subtask:
Filter the main DataFrame (df) to include only records for the state of 'ohio' and the specified 'key_year' (2024).


**Reasoning**:
To filter the DataFrame `df` for records corresponding to 'ohio' and the `key_year`, I will use boolean indexing on both conditions and store the result in `OHkeyyear`.



In [ ]:
OHkeyyear = AIkeyyear[(df['state'] == 'ohio') & (df['year'] == key_year)]
print(f"The 'df' DataFrame has been filtered for the state of ohio and the year {key_year}, resulting in {len(OHkeyyear)} rows.")

The 'df' DataFrame has been filtered for the state of ohio and the year 2024, resulting in 36484 rows.


## Clean and Capitalize Ohio Data

### Subtask:
Create a copy of the filtered Ohio data, clean and standardize operator, field, and formation names, and convert geographic coordinates.


**Reasoning**:
To prepare the Ohio data for analysis and GOGET bulk upload, I will create a copy of the filtered Ohio DataFrame, convert text columns to title case, handle missing values in field and formation names, standardize geographic coordinates to numeric, and replace 'WC' with 'Wild cat' in field names for consistency.



In [ ]:
ohio_df_capitalized = OHkeyyear.copy()
ohio_df_capitalized['operator_name'] = ohio_df_capitalized['operator_name'].str.title()
ohio_df_capitalized['field_name'] = ohio_df_capitalized['field_name'].str.title()
ohio_df_capitalized['formation_name'] = ohio_df_capitalized['formation_name'].str.title()

ohio_df_capitalized['longitude'] = pd.to_numeric(ohio_df_capitalized['longitude'], errors='coerce')
ohio_df_capitalized['latitude'] = pd.to_numeric(ohio_df_capitalized['latitude'], errors='coerce')

ohio_df_capitalized['formation_name'] = ohio_df_capitalized['formation_name'].fillna('Formation not assigned')
ohio_df_capitalized['field_name'] = ohio_df_capitalized['field_name'].fillna('Field not assigned')

ohio_df_capitalized['field_name'] = ohio_df_capitalized['field_name'].str.replace('WC', 'Wild cat', regex=False)

print("Ohio data has been cleaned and standardized.")

Ohio data has been cleaned and standardized.


**Reasoning**:
The previous step cleaned and standardized the Ohio data. The next logical step, according to the overall task, is to group the data by `field_name` and `operator_name` to aggregate production volumes and other relevant details, similar to how the Wyoming data was processed.



In [ ]:
ohio_df_grouped_field = ohio_df_capitalized.groupby(['field_name', 'operator_name']).agg({
    'well_no': lambda x: '; '.join(x.astype(str).unique()),
    'longitude': 'mean',
    'latitude': 'mean',
    'gross_gas_vol': 'sum',
    'dry_gas_vol': 'sum',
    'oil_and_condensate_vol': 'sum',
    'ngl_vol': 'sum',
    'formation_name': lambda x: '; '.join(x.astype(str).unique()),
    'resource_type': lambda x: '; '.join(x.astype(str).unique())
}).reset_index()

print(f'Ohio has {len(ohio_df_grouped_field)} rows when grouped by field + operator')
print(f'Ohio has {len(ohio_df_grouped_field[ohio_df_grouped_field["field_name"] == "Field not assigned"])} rows with "Field not assigned"')


Ohio has 683 rows when grouped by field + operator
Ohio has 683 rows with "Field not assigned"


**Reasoning**:
The previous grouping by `field_name` and `operator_name` unexpectedly resulted in an empty DataFrame. To proceed with the overall task of grouping by both field and formation, and to further diagnose the issue, the next logical step is to perform the grouping by `formation_name` and `operator_name` as specified in the task, similar to how it was done for Wyoming data. This will help determine if the problem is specific to the 'field_name' grouping or a more general issue with the Ohio data for grouping.



In [ ]:
ohio_df_grouped_formation = ohio_df_capitalized.groupby(['formation_name', 'operator_name']).agg({
    'well_no': lambda x: '; '.join(x.astype(str).unique()),
    'longitude': 'mean',
    'latitude': 'mean',
    'gross_gas_vol': 'sum',
    'dry_gas_vol': 'sum',
    'oil_and_condensate_vol': 'sum',
    'ngl_vol': 'sum',
    'resource_type': lambda x: '; '.join(x.astype(str).unique())
}).reset_index()

print(f'Ohio has {len(ohio_df_grouped_formation)} rows when grouped by formation + operator')
print(f'Ohio has {len(ohio_df_grouped_formation[ohio_df_grouped_formation["formation_name"] == "Formation not assigned"])} rows with "Formation not assigned"')

Ohio has 2371 rows when grouped by formation + operator
Ohio has 254 rows with "Formation not assigned"


**Reasoning**:
Both previous grouping attempts (by field and by formation) resulted in empty DataFrames, which is unexpected. To diagnose this, I need to inspect the `ohio_df_capitalized` DataFrame, specifically the columns used for grouping and aggregation, to identify why no groups are being formed. I will display the head of the DataFrame and check for nulls in critical columns.



In [ ]:
print("Head of ohio_df_capitalized:")
display(ohio_df_capitalized.head())

print("Checking for nulls in grouping and aggregation columns:")
print(ohio_df_capitalized[['field_name', 'operator_name', 'formation_name', 'gross_gas_vol', 'dry_gas_vol', 'oil_and_condensate_vol', 'ngl_vol']].isnull().sum())

Head of ohio_df_capitalized:


,country,state,county_name,year,well_no,well_name,field_no,field_name,lease_no,lease_name,...,resource_type,resource_type_detail,recovery_mechanism,field_terrain,oil_gravity,gas_gravity,oil_and_condensate_vol,gross_gas_vol,dry_gas_vol,ngl_vol
3957002,united_states,ohio,NaN,2024.0,3.4005233e+13,None,None,Field not assigned,None,CHESROWN DOYLE & SHIRLEY,...,Conventional,None,None,Onshore,NaN,NaN,0.000013,0.000000,0.000000,0.000000
3957012,united_states,ohio,NaN,2024.0,3.4005237e+13,None,None,Field not assigned,None,SCHWEDER,...,Conventional,None,None,Onshore,NaN,NaN,0.000025,0.000000,0.000000,0.000000
3957030,united_states,ohio,NaN,2024.0,3.4005241e+13,None,None,Field not assigned,None,YOUNG,...,Conventional,None,None,Onshore,NaN,NaN,0.000013,0.000000,0.000000,0.000000
3957047,united_states,ohio,NaN,2024.0,3.4007203e+13,None,None,Field not assigned,None,LESKO ANDREW & ALBINA,...,Conventional,None,None,Onshore,NaN,NaN,0.000000,0.041144,0.039754,0.000034
3957057,united_states,ohio,NaN,2024.0,3.4007208e+13,None,None,Field not assigned,None,HELFER HAROLD D,...,Conventional,None,None,Onshore,NaN,NaN,0.000000,0.008863,0.008564,0.000007


Checking for nulls in grouping and aggregation columns:
field_name                 0
operator_name             12
formation_name             0
gross_gas_vol              0
dry_gas_vol                0
oil_and_condensate_vol     0
ngl_vol                    0
dtype: int64


**Reasoning**:
The previous output showed that the 'operator_name' column in `ohio_df_capitalized` is entirely null, causing the groupby operations to return empty DataFrames. To enable grouping by operator, I need to fill these null values with a placeholder, such as 'Operator not assigned'.



## California

In [ ]:
# set year and state

CAkeyyear = AIkeyyear[(AIkeyyear['state'] == 'california')]
print(f"The 'df' DataFrame has been filtered for the state of california and the year {key_year}, resulting in {len(CAkeyyear)} rows.")

The 'df' DataFrame has been filtered for the state of california and the year 2024, resulting in 54978 rows.


In [ ]:
# clean names and fix capitalization
CA_df_capitalized = CAkeyyear.copy()

CA_df_capitalized['operator_name'] = CA_df_capitalized['operator_name'].str.title()
CA_df_capitalized['field_name'] = CA_df_capitalized['field_name'].str.title()
CA_df_capitalized['formation_name'] = CA_df_capitalized['formation_name'].str.title()

CA_df_capitalized['longitude'] = pd.to_numeric(CA_df_capitalized['longitude'], errors='coerce')
CA_df_capitalized['latitude'] = pd.to_numeric(CA_df_capitalized['latitude'], errors='coerce')

CA_df_capitalized['formation_name'] = CA_df_capitalized['formation_name'].fillna('Formation not assigned')
CA_df_capitalized['field_name'] = CA_df_capitalized['field_name'].fillna('Field not assigned')

CA_df_capitalized['field_name'] = CA_df_capitalized['field_name'].str.replace('Wc', 'Wild cat', regex=False)

CA_df_capitalized['operator_name'] = CA_df_capitalized['operator_name'].fillna('Operator not assigned')

print("California data has been cleaned and standardized.")

California data has been cleaned and standardized.


In [ ]:
CA_df_capitalized.columns

Index(['country', 'state', 'county_name', 'year', 'well_no', 'well_name',
       'field_no', 'field_name', 'lease_no', 'lease_name', 'formation_no',
       'formation_name', 'longitude', 'latitude', 'operator_no',
       'operator_name', 'owner_no', 'owner_name', 'ownership_perc',
       'resource_type', 'resource_type_detail', 'recovery_mechanism',
       'field_terrain', 'oil_gravity', 'gas_gravity', 'oil_and_condensate_vol',
       'gross_gas_vol', 'dry_gas_vol', 'ngl_vol'],
      dtype='object')

In [ ]:
# check entries where Field_name = Elk Hills, operator_name = California Resources
Elkhills_CR = CA_df_capitalized[(CA_df_capitalized['field_name'] == 'Elk Hills') & (CA_df_capitalized['operator_name'] == 'California Resources Elk Hills  Llc')]

# Previous data was more granular

In [ ]:
# group by field
CA_df_grouped_field = CA_df_capitalized.groupby(['field_name', 'operator_name']).agg({
    'well_no': lambda x: '; '.join(x.astype(str).unique()),
    'longitude': 'mean',
    'latitude': 'mean',
    'gross_gas_vol': 'sum',
    'dry_gas_vol': 'sum',
    'oil_and_condensate_vol': 'sum',
    'ngl_vol': 'sum',
    'formation_name': lambda x: '; '.join(x.astype(str).unique()),
    'resource_type': lambda x: '; '.join(x.astype(str).unique())
}).reset_index()

print(f'California has {len(CA_df_grouped_field)} rows when grouped by field + operator')
print(f'California has {len(CA_df_grouped_field[CA_df_grouped_field["field_name"] == "Field not assigned"])} rows with "Field not assigned"')

California has 569 rows when grouped by field + operator
California has 0 rows with "Field not assigned"


In [ ]:
# group by formation
CA_df_grouped_formation = CA_df_capitalized.groupby(['formation_name', 'operator_name']).agg({
    'well_no': lambda x: '; '.join(x.astype(str).unique()),
    'longitude': 'mean',
    'latitude': 'mean',
    'gross_gas_vol': 'sum',
    'dry_gas_vol': 'sum',
    'oil_and_condensate_vol': 'sum',
    'ngl_vol': 'sum',
    'resource_type': lambda x: '; '.join(x.astype(str).unique())
}).reset_index()

print(f'California has {len(CA_df_grouped_formation)} rows when grouped by formation + operator')
print(f'California has {len(CA_df_grouped_formation[CA_df_grouped_formation["formation_name"] == "Formation not assigned"])} rows with "Formation not assigned"')

California has 250 rows when grouped by formation + operator
California has 250 rows with "Formation not assigned"


In [ ]:
# check totals
CA_dry_gas_vol_sum_form = CA_df_grouped_formation['dry_gas_vol'].sum()
CA_gross_gas_vol_sum_form = CA_df_grouped_formation['gross_gas_vol'].sum()
CA_oil_and_condensate_vol_sum_form = CA_df_grouped_formation['oil_and_condensate_vol'].sum()
CA_ngl_vol_sum_form = CA_df_grouped_formation['ngl_vol'].sum()

CA_dry_gas_vol_sum_fied = CA_df_grouped_field['dry_gas_vol'].sum()
CA_gross_gas_vol_sum_field = CA_df_grouped_field['gross_gas_vol'].sum()
CA_oil_and_condensate_vol_sum_field = CA_df_grouped_field['oil_and_condensate_vol'].sum()
CA_ngl_vol_sum_field = CA_df_grouped_field['ngl_vol'].sum()

print(f'Formation grouped dry gas sum = {CA_dry_gas_vol_sum_form}')
print(f'Formation grouped gross gas sum = {CA_gross_gas_vol_sum_form}')
print(f'Formation grouped oil and condensate sum = {CA_oil_and_condensate_vol_sum_form}')
print(f'Formation grouped ngl sum = {CA_ngl_vol_sum_form}')

print(f'Field grouped dry gas sum = {CA_dry_gas_vol_sum_fied}')
print(f'Field grouped gross gas sum = {CA_gross_gas_vol_sum_field}')
print(f'Field grouped oil and condensate sum = {CA_oil_and_condensate_vol_sum_field}')
print(f'Field grouped ngl sum = {CA_ngl_vol_sum_field}')

Formation grouped dry gas sum = 3210.686282581366
Formation grouped gross gas sum = 3414.2081821436013
Formation grouped oil and condensate sum = 112.5979743808689
Formation grouped ngl sum = 4.452663431049057
Field grouped dry gas sum = 3210.6862825813664
Field grouped gross gas sum = 3414.2081821436013
Field grouped oil and condensate sum = 112.59797438086889
Field grouped ngl sum = 4.452663431049056


In [ ]:
# parse operators, to add in Entity ID's manually
# calculate total production
CA_df_grouped_field['total_hydrocarbons_production'] = (CA_df_grouped_field['dry_gas_vol']*0.005883) + CA_df_grouped_field['oil_and_condensate_vol']+ CA_df_grouped_field['ngl_vol']
CA_df_grouped_formation['total_hydrocarbons_production'] =  (CA_df_grouped_formation['dry_gas_vol']*0.005883) + CA_df_grouped_formation['oil_and_condensate_vol'] + CA_df_grouped_formation['ngl_vol']

# group by operator for datacleaning
CA_df_grouped_field_op = CA_df_grouped_field.groupby('operator_name').agg({'total_hydrocarbons_production': 'sum'})

# sort by production high to low
CA_df_grouped_field_op =CA_df_grouped_field_op.sort_values(by='total_hydrocarbons_production', ascending=False)

CA_GOGET_Folder = '/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/Data sets & reports (O&G)/O&G country materials/United States (GOGET O&G)/California (GOGET)'
CA_df_grouped_field_op.to_csv(f'{CA_GOGET_Folder}/CA_grouped_field_op.csv')


In [ ]:
# load in operator and discovery data (manually gathered)
CA_top_op = pd.read_csv(f'{CA_GOGET_Folder}/CA_grouped_field_top_op - CA_grouped_field_op.csv')
CA_disco_years = pd.read_csv(f'{CA_GOGET_Folder}/California Field Code and Discovery Year - Units in GOGET.csv')

print("California operator entity IDs and field discovery years have been loaded.")

California operator entity IDs and field discovery years have been loaded.


In [ ]:
# update based on top operator cut offs
CA_top_op_for_merge = CA_top_op[['operator_name', 'Entity ID']].copy()

# merge Entity ID based on operator_name
CA_Form_ID = CA_df_grouped_formation.merge(CA_top_op_for_merge, on='operator_name', how='left')
CA_Field_ID = CA_df_grouped_field.merge(CA_top_op_for_merge, on='operator_name', how='left')

# drop rows where Entity ID is null
CA_Form_ID_noNull = CA_Form_ID.dropna(subset=['Entity ID']).copy()
CA_Field_ID_noNull = CA_Field_ID.dropna(subset=['Entity ID']).copy()

# print row count pre and post drop formation and field
print(f'Formation grouped has {len(CA_df_grouped_formation)} rows')
print(f'Formation groups with only top ops has {len(CA_Form_ID_noNull)} rows')
print(f'Field grouped has {len(CA_df_grouped_field)} rows')
print(f'Field groups with only top ops has {len(CA_Field_ID_noNull)} rows')


Formation grouped has 250 rows
Formation groups with only top ops has 25 rows
Field grouped has 569 rows
Field groups with only top ops has 205 rows


In [ ]:
row_choice = 'field' # User can change this to 'formation'

if row_choice == 'formation':
    choice = CA_Form_ID_noNull
    name_col = 'formation_name'
    project_complex_col = 'formation_name'
elif row_choice == 'field':
    choice = CA_Field_ID_noNull
    name_col = 'field_name'
    project_complex_col = 'formation_name'
else:
    raise ValueError("Invalid row_choice. Must be 'formation' or 'field'.")

print(f'Choice is {row_choice}')
choice.head()

Choice is field


,field_name,operator_name,well_no,longitude,latitude,gross_gas_vol,dry_gas_vol,oil_and_condensate_vol,ngl_vol,formation_name,resource_type,total_hydrocarbons_production,Entity ID
0,Aliso Canyon,Crimson Resource Management Corp.,40370077201,NaN,NaN,0.123093,0.115756,0.005710,0.000161,Formation not assigned,Conventional,0.006552,E100000192133
3,Ant Hill,California Resources Production Corporation,40290400501; 40302063700; 40302260000; 4030311...,-118.856771,35.401848,0.065242,0.061353,0.013736,0.000085,Formation not assigned,Conventional; Unconventional,0.014182,E100000001169
4,Antelope Hills,E & B Natural Resources Management Corporation,40290132500; 40290132600; 40291343700; 4029134...,-119.856074,35.519493,0.196774,0.185044,0.038637,0.000257,Formation not assigned,Conventional; Unconventional,0.039982,E100002015600
5,Antelope Hills North,California Resources Production Corporation,40294584300; 40297259100; 40297259200; 4029725...,-119.907953,35.576792,0.000000,0.000000,0.056052,0.000000,Formation not assigned,Unconventional,0.056052,E100000001169
6,Antelope Hills North,E & B Natural Resources Management Corporation,40291351200; 40291351700; 40291351900; 4029135...,-119.890227,35.562973,0.280195,0.263493,0.006151,0.000365,Formation not assigned,Conventional; Unconventional,0.008067,E100002015600


In [ ]:
# standardize the 'resource_type' column for GOGET bulk upload

choice['resource_type'] = choice['resource_type'].replace({
    'Unconventional; Conventional': 'Unconventional',
    'Conventional; Unconventional': 'Unconventional',
    'Conventional': 'Conventional'
})

print("Resource types standardized.")

Resource types standardized.


In [ ]:
# define fuel type
import numpy as np

conditions = [
    (choice['dry_gas_vol'] > 0) & (choice['oil_and_condensate_vol'] > 0),
    (choice['dry_gas_vol'] == 0) & (choice['oil_and_condensate_vol'] > 0),
    (choice['dry_gas_vol'] > 0) & (choice['oil_and_condensate_vol'] == 0)
]

choices = ['oil and gas', 'oil', 'gas']

choice['fuel_type'] = np.select(conditions, choices, default=None)

print("Fuel types have been assigned.")

Fuel types have been assigned.


In [ ]:
choice = choice.merge(CA_disco_years, left_on=name_col, right_on='Unit Name', how='left')
choice = choice.drop(columns=['Unit Name'])

print("Discovery years merged.")

Discovery years merged.


In [ ]:
# format names based on field or formation list - add source
choice.loc[:, 'name'] = choice[name_col] + ' - ' + choice['operator_name']
choice.loc[:, 'unit_type'] = "asset"
choice.loc[:, 'project_complex'] = choice[project_complex_col]
choice.loc[:, 'coordinate_accuracy'] = "exact"
choice.loc[:, 'operator_source'] = "DOGGR_Op_2025"
choice.loc[:, 'location_source'] = "DOGGR_Loc_2025; DOGGR_Prod_2024"
choice.loc[:, 'resource_type_source'] = "DOGGR_Prod_2024"
choice.loc[:, 'discovery_year_source'] = "DOGGR_Prod_2024"
choice['name'] = choice['name'].str.replace('Wc', 'Wildcat', regex=False)

print("Data formatted for GOGET bulk upload.")

Data formatted for GOGET bulk upload.


In [ ]:
# add total production column if it doesn't already exist (it should from the earlier step, but this is a safeguard)
if 'total_hydrocarbons_production' not in choice.columns:
    choice['total_hydrocarbons_production'] = (choice['dry_gas_vol']*0.005883) + choice['oil_and_condensate_vol'] + choice['ngl_vol']

# Select and order the final columns for CA_Clean
CA_Clean = choice[[
    'name',
    'unit_type',
    'fuel_type',
    'project_complex',
    'longitude',
    'latitude',
    'coordinate_accuracy',
    'location_source',
    'operator_name',
    'operator_source',
    'Entity ID',
    'resource_type',
    'resource_type_source',
    'Discovery year',
    'discovery_year_source',
    'well_no',
    'total_hydrocarbons_production'
]].copy()

# sort by total_hydrocarbons_production high to low
CA_Clean = CA_Clean.sort_values(by='total_hydrocarbons_production', ascending=False).reset_index(drop=True)

print("CA_Clean DataFrame created with selected and ordered columns.")
display(CA_Clean.head())

CA_Clean DataFrame created with selected and ordered columns.


,name,unit_type,fuel_type,project_complex,longitude,latitude,coordinate_accuracy,location_source,operator_name,operator_source,Entity ID,resource_type,resource_type_source,Discovery year,discovery_year_source,well_no,total_hydrocarbons_production
0,Elk Hills - California Resources Elk Hills Llc,asset,oil and gas,Formation not assigned,-119.458095,35.281828,exact,DOGGR_Loc_2025; DOGGR_Prod_2024,California Resources Elk Hills Llc,DOGGR_Op_2025,E100001010206,Unconventional,DOGGR_Prod_2024,1919.0,DOGGR_Prod_2024,4.0302e+10; 40290717800; 40292703700; 40292704...,16.458852
1,Belridge South - Aera Energy Llc,asset,oil and gas,Formation not assigned,-119.717119,35.453560,exact,DOGGR_Loc_2025; DOGGR_Prod_2024,Aera Energy Llc,DOGGR_Op_2025,E100000002366,Unconventional,DOGGR_Prod_2024,1911.0,DOGGR_Prod_2024,40293337700; 40294598600; 40294713700; 4029520...,13.303380
2,Kern River - Chevron U.S.A. Inc.,asset,oil and gas,Formation not assigned,-118.988749,35.447414,exact,DOGGR_Loc_2025; DOGGR_Prod_2024,Chevron U.S.A. Inc.,DOGGR_Op_2025,E100000001179,Unconventional,DOGGR_Prod_2024,1899.0,DOGGR_Prod_2024,4.0301e+10; 40290000100; 40290003800; 40290003...,13.016367
3,Cymric - Chevron U.S.A. Inc.,asset,oil and gas,Formation not assigned,-119.661640,35.349810,exact,DOGGR_Loc_2025; DOGGR_Prod_2024,Chevron U.S.A. Inc.,DOGGR_Op_2025,E100000001179,Unconventional,DOGGR_Prod_2024,1916.0,DOGGR_Prod_2024,40290333700; 40290335300; 40290342100; 4029034...,6.385726
4,Midway-Sunset - Berry Petroleum Company Llc,asset,oil and gas,Formation not assigned,-119.497457,35.168779,exact,DOGGR_Loc_2025; DOGGR_Prod_2024,Berry Petroleum Company Llc,DOGGR_Op_2025,E100002008419,Unconventional,DOGGR_Prod_2024,1901.0,DOGGR_Prod_2024,40290940000; 40290940100; 40291195600; 4029119...,4.981904


In [ ]:
# check totals in choice
CA_choice_dry_gas = choice['dry_gas_vol'].sum()
CA_choice_gross_gas = choice['gross_gas_vol'].sum()
CA_choice_condensate = choice['oil_and_condensate_vol'].sum()
CA_choice_ngl = choice['ngl_vol'].sum()

# print totals
print(f'{row_choice} top op dry gas sum = {CA_choice_dry_gas}')
print(f'{row_choice} top op gross gas sum = {CA_choice_gross_gas}')
print(f'{row_choice} top op oil and condensate sum = {CA_choice_condensate}')
print(f'{row_choice} top op  NGL sum = {CA_choice_ngl}')

field top op dry gas sum = 3094.3396870019537
field top op gross gas sum = 3290.486502841937
field top op oil and condensate sum = 108.57008812039527
field top op  NGL sum = 4.291310939441875


In [ ]:
GOGET_IDs = pd.read_csv(f'{CA_GOGET_Folder}/GOGET CA - 2025-12-15T212103 - Unit ids.csv')

goget_unit_names = GOGET_IDs['Unit Name'].tolist()

def fuzzy_match_unit_id(row, choices, ids_df, threshold=87): #70 was way too low, 85 still too low, 86 had 205 matches, 87 had 4 - 2025 01 21
    match = process.extractOne(row['name'], choices, score_cutoff=threshold)
    if match:
        original_unit_name = match[0]
        unit_id = ids_df[ids_df['Unit Name'] == original_unit_name]['Unit ID'].iloc[0]
        return unit_id
    return np.nan

CA_Clean['GOGET_Unit_ID'] = CA_Clean.apply(fuzzy_match_unit_id, axis=1, choices=goget_unit_names, ids_df=GOGET_IDs)

matches = len(CA_Clean[CA_Clean['GOGET_Unit_ID'].notna()])

print("Fuzzy matching complete.")
print(f"{matches}" +' matches were made')
print("Displaying first 5 rows with new GOGET_Unit_ID:")
display(CA_Clean.head())

Fuzzy matching complete.
4 matches were made
Displaying first 5 rows with new GOGET_Unit_ID:


,name,unit_type,fuel_type,project_complex,longitude,latitude,coordinate_accuracy,location_source,operator_name,operator_source,Entity ID,resource_type,resource_type_source,Discovery year,discovery_year_source,well_no,total_hydrocarbons_production,GOGET_Unit_ID
0,Elk Hills - California Resources Elk Hills Llc,asset,oil and gas,Formation not assigned,-119.458095,35.281828,exact,DOGGR_Loc_2025; DOGGR_Prod_2024,California Resources Elk Hills Llc,DOGGR_Op_2025,E100001010206,Unconventional,DOGGR_Prod_2024,1919.0,DOGGR_Prod_2024,4.0302e+10; 40290717800; 40292703700; 40292704...,16.458852,NaN
1,Belridge South - Aera Energy Llc,asset,oil and gas,Formation not assigned,-119.717119,35.453560,exact,DOGGR_Loc_2025; DOGGR_Prod_2024,Aera Energy Llc,DOGGR_Op_2025,E100000002366,Unconventional,DOGGR_Prod_2024,1911.0,DOGGR_Prod_2024,40293337700; 40294598600; 40294713700; 4029520...,13.303380,NaN
2,Kern River - Chevron U.S.A. Inc.,asset,oil and gas,Formation not assigned,-118.988749,35.447414,exact,DOGGR_Loc_2025; DOGGR_Prod_2024,Chevron U.S.A. Inc.,DOGGR_Op_2025,E100000001179,Unconventional,DOGGR_Prod_2024,1899.0,DOGGR_Prod_2024,4.0301e+10; 40290000100; 40290003800; 40290003...,13.016367,NaN
3,Cymric - Chevron U.S.A. Inc.,asset,oil and gas,Formation not assigned,-119.661640,35.349810,exact,DOGGR_Loc_2025; DOGGR_Prod_2024,Chevron U.S.A. Inc.,DOGGR_Op_2025,E100000001179,Unconventional,DOGGR_Prod_2024,1916.0,DOGGR_Prod_2024,40290333700; 40290335300; 40290342100; 4029034...,6.385726,NaN
4,Midway-Sunset - Berry Petroleum Company Llc,asset,oil and gas,Formation not assigned,-119.497457,35.168779,exact,DOGGR_Loc_2025; DOGGR_Prod_2024,Berry Petroleum Company Llc,DOGGR_Op_2025,E100002008419,Unconventional,DOGGR_Prod_2024,1901.0,DOGGR_Prod_2024,40290940000; 40290940100; 40291195600; 4029119...,4.981904,NaN


In [ ]:
production_cols = ['gross_gas_vol', 'dry_gas_vol', 'oil_and_condensate_vol', 'ngl_vol']
id_cols = [col for col in choice.columns if col not in production_cols]

CA_long_prod = pd.melt(choice,
                                  id_vars=id_cols,
                                  value_vars=production_cols,
                                  var_name='production_type',
                                  value_name='volume')

cols_to_drop_from_melted = ['Entity ID','well_no','longitude','latitude','unit_type', 'project_complex','coordinate_accuracy','operator_source','location_source', 'field_name','resource_type','resource_type_source','fuel_type','discovery_year_source','Discovery year', 'operator_name', 'total_hydrocarbons_production']

# Dynamically add 'formation_name' if it was not used as name_col
if row_choice == 'field': # if row_choice is 'field', formation_name is still a column and needs to be dropped
    if 'formation_name' in CA_long_prod.columns:
        cols_to_drop_from_melted.append('formation_name')

existing_cols_to_drop = [col for col in cols_to_drop_from_melted if col in CA_long_prod.columns]
CA_long_prod.drop(columns=existing_cols_to_drop, inplace=True)

print("California production data melted into long format.")
display(CA_long_prod.head())

California production data melted into long format.


,name,production_type,volume
0,Aliso Canyon - Crimson Resource Management Corp.,gross_gas_vol,0.123093
1,Ant Hill - California Resources Production Cor...,gross_gas_vol,0.065242
2,Antelope Hills - E & B Natural Resources Manag...,gross_gas_vol,0.196774
3,Antelope Hills North - California Resources P...,gross_gas_vol,0.000000
4,Antelope Hills North - E & B Natural Resource...,gross_gas_vol,0.280195


In [ ]:
CA_long_prod['fuel description'] = CA_long_prod['production_type'].replace({'gross_gas_vol': 'gross gas', 'dry_gas_vol': 'dry gas', 'oil_and_condensate_vol': 'oil and condensates', 'ngl_vol': 'NGL'})

CA_long_prod['Year'] = key_year

CA_long_prod['production/reserves'] = 'production'

CA_long_prod.rename(columns={'volume': 'Quantity'}, inplace=True)

CA_long_prod['quantity_unit'] = CA_long_prod['fuel description'].apply(lambda x: 'million cubic meters/y' if x in ['gross gas', 'dry gas'] else 'million bbl/y')

CA_long_prod['Datasource'] = 'DOGGR_Prod_2024'

print("CA_long_prod has been formatted for GOGET bulk upload.")
display(CA_long_prod.head())

CA_long_prod has been formatted for GOGET bulk upload.


,name,production_type,Quantity,fuel description,Year,production/reserves,quantity_unit,Datasource
0,Aliso Canyon - Crimson Resource Management Corp.,gross_gas_vol,0.123093,gross gas,2024,production,million cubic meters/y,DOGGR_Prod_2024
1,Ant Hill - California Resources Production Cor...,gross_gas_vol,0.065242,gross gas,2024,production,million cubic meters/y,DOGGR_Prod_2024
2,Antelope Hills - E & B Natural Resources Manag...,gross_gas_vol,0.196774,gross gas,2024,production,million cubic meters/y,DOGGR_Prod_2024
3,Antelope Hills North - California Resources P...,gross_gas_vol,0.000000,gross gas,2024,production,million cubic meters/y,DOGGR_Prod_2024
4,Antelope Hills North - E & B Natural Resource...,gross_gas_vol,0.280195,gross gas,2024,production,million cubic meters/y,DOGGR_Prod_2024


In [ ]:
import datetime
time_stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# try saving as an excel instead
CA_Clean.to_excel(f'{CA_GOGET_Folder}/CA_AI_FOR_GOGET{key_year}_{time_stamp}.xlsx', index=False)
# add CA_long_prod as a tab to existing excel CA_Clean
with pd.ExcelWriter(f'{CA_GOGET_Folder}/CA_AI_FOR_GOGET{key_year}_{time_stamp}.xlsx', engine='openpyxl', mode='a') as writer:
    CA_long_prod.to_excel(writer, sheet_name='long_prod', index=False)
    print(f'Saved CA_AI_FOR_GOGET{key_year}_{time_stamp}.xlsx with long_prod tab')

Saved CA_AI_FOR_GOGET2024_2026-01-21_21-11-04.xlsx with long_prod tab


## Texas

In [ ]:
# set year and state
key_year = 2024
active_state = 'texas'
activestatekeyyear = df[(df['state'] == active_state) & (df['year'] == key_year)]
print(f"The 'df' DataFrame has been filtered for the state of {active_state} and the year {key_year}, resulting in {len(activestatekeyyear)} rows.")

The 'df' DataFrame has been filtered for the state of texas and the year 2024, resulting in 155212 rows.


In [ ]:
# clean names and fix capitalization
ASKY_df_capitalized = activestatekeyyear.copy()

ASKY_df_capitalized['operator_name'] = ASKY_df_capitalized['operator_name'].str.title()
ASKY_df_capitalized['field_name'] = ASKY_df_capitalized['field_name'].str.title()
ASKY_df_capitalized['formation_name'] = ASKY_df_capitalized['formation_name'].str.title()

ASKY_df_capitalized['longitude'] = pd.to_numeric(ASKY_df_capitalized['longitude'], errors='coerce')
ASKY_df_capitalized['latitude'] = pd.to_numeric(ASKY_df_capitalized['latitude'], errors='coerce')

ASKY_df_capitalized['formation_name'] = ASKY_df_capitalized['formation_name'].fillna('Formation not assigned')
ASKY_df_capitalized['field_name'] = ASKY_df_capitalized['field_name'].fillna('Field not assigned')

ASKY_df_capitalized['field_name'] = ASKY_df_capitalized['field_name'].str.replace('Wc', 'Wild cat', regex=False)

ASKY_df_capitalized['operator_name'] = ASKY_df_capitalized['operator_name'].fillna('Operator not assigned')

print(f"{active_state} data has been cleaned and standardized.")

texas data has been cleaned and standardized.


In [ ]:
# group by field
ASKY_df_grouped_field = ASKY_df_capitalized.groupby(['field_name', 'operator_name']).agg({
    'well_no': lambda x: '; '.join(x.astype(str).unique()),
    'longitude': 'mean',
    'latitude': 'mean',
    'gross_gas_vol': 'sum',
    'dry_gas_vol': 'sum',
    'oil_and_condensate_vol': 'sum',
    'ngl_vol': 'sum',
    'formation_name': lambda x: '; '.join(x.astype(str).unique()),
    'resource_type': lambda x: '; '.join(x.astype(str).unique())
}).reset_index()

print(f'{active_state} has {len(ASKY_df_grouped_field)} rows when grouped by field + operator')
print(f'{active_state} has {len(ASKY_df_grouped_field[ASKY_df_grouped_field["field_name"] == "Field not assigned"])} rows with "Field not assigned"')

texas has 22078 rows when grouped by field + operator
texas has 1 rows with "Field not assigned"


In [ ]:
# group by formation
ASKY_df_grouped_formation = ASKY_df_capitalized.groupby(['formation_name', 'operator_name']).agg({
    'well_no': lambda x: '; '.join(x.astype(str).unique()),
    'longitude': 'mean',
    'latitude': 'mean',
    'gross_gas_vol': 'sum',
    'dry_gas_vol': 'sum',
    'oil_and_condensate_vol': 'sum',
    'ngl_vol': 'sum',
    'resource_type': lambda x: '; '.join(x.astype(str).unique())
}).reset_index()

print(f'{active_state} has {len(ASKY_df_grouped_formation)} rows when grouped by formation + operator')
print(f'{active_state} has {len(ASKY_df_grouped_formation[ASKY_df_grouped_formation["formation_name"] == "Formation not assigned"])} rows with "Formation not assigned"')

texas has 16561 rows when grouped by formation + operator
texas has 1789 rows with "Formation not assigned"


In [ ]:
# check totals
ASKY_dry_gas_vol_sum_form = ASKY_df_grouped_formation['dry_gas_vol'].sum()
ASKY_gross_gas_vol_sum_form = ASKY_df_grouped_formation['gross_gas_vol'].sum()
ASKY_oil_and_condensate_vol_sum_form = ASKY_df_grouped_formation['oil_and_condensate_vol'].sum()
ASKY_ngl_vol_sum_form = ASKY_df_grouped_formation['ngl_vol'].sum()

ASKY_dry_gas_vol_sum_field = ASKY_df_grouped_field['dry_gas_vol'].sum()
ASKY_gross_gas_vol_sum_field = ASKY_df_grouped_field['gross_gas_vol'].sum()
ASKY_oil_and_condensate_vol_sum_field = ASKY_df_grouped_field['oil_and_condensate_vol'].sum()
ASKY_ngl_vol_sum_field = ASKY_df_grouped_field['ngl_vol'].sum()

# check if sums are equal in both formation and field
if ASKY_dry_gas_vol_sum_form == ASKY_dry_gas_vol_sum_field and ASKY_gross_gas_vol_sum_form == ASKY_gross_gas_vol_sum_field and ASKY_oil_and_condensate_vol_sum_form == ASKY_oil_and_condensate_vol_sum_field and ASKY_ngl_vol_sum_form == ASKY_ngl_vol_sum_field: print('sums match')
else: print('sums do not match')

print(f'Formation grouped dry gas sum = {ASKY_dry_gas_vol_sum_form}')
print(f'Formation grouped gross gas sum = {ASKY_gross_gas_vol_sum_form}')
print(f'Formation grouped oil and condensate sum = {ASKY_oil_and_condensate_vol_sum_form}')
print(f'Formation grouped ngl sum = {ASKY_ngl_vol_sum_form}')

print(f'Field grouped dry gas sum = {ASKY_dry_gas_vol_sum_field}')
print(f'Field grouped gross gas sum = {ASKY_gross_gas_vol_sum_field}')
print(f'Field grouped oil and condensate sum = {ASKY_oil_and_condensate_vol_sum_field}')
print(f'Field grouped ngl sum = {ASKY_ngl_vol_sum_field}')

sums do not match
Formation grouped dry gas sum = 288552.26413919014
Formation grouped gross gas sum = 374899.2217922962
Formation grouped oil and condensate sum = 2062.99726
Formation grouped ngl sum = 824.2709187519883
Field grouped dry gas sum = 288552.2641391901
Field grouped gross gas sum = 374899.22179229616
Field grouped oil and condensate sum = 2062.99726
Field grouped ngl sum = 824.2709187519883


In [ ]:
# parse operators, to add in Entity ID's manually
# calculate total production
ASKY_df_grouped_field['total_hydrocarbons_production'] = (ASKY_df_grouped_field['dry_gas_vol']*0.005883) + ASKY_df_grouped_field['oil_and_condensate_vol']+ ASKY_df_grouped_field['ngl_vol']
ASKY_df_grouped_formation['total_hydrocarbons_production'] =  (ASKY_df_grouped_formation['dry_gas_vol']*0.005883) + ASKY_df_grouped_formation['oil_and_condensate_vol'] + ASKY_df_grouped_formation['ngl_vol']

# group by operator for datacleaning
ASKY_df_grouped_field_op = ASKY_df_grouped_field.groupby('operator_name').agg({'total_hydrocarbons_production': 'sum'})

# sort by production high to low
ASKY_df_grouped_field_op =ASKY_df_grouped_field_op.sort_values(by='total_hydrocarbons_production', ascending=False)

TX_GOGET_Folder = '/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/Data sets & reports (O&G)/O&G country materials/United States (GOGET O&G)/Texas (GOGET)'
ASKY_df_grouped_field_op.to_csv(f'{TX_GOGET_Folder}/TX_grouped_field_op.csv')


In [ ]:
# load in operator data (manually gathered)
ASKY_top_op = pd.read_csv(f'{TX_GOGET_Folder}/TX_grouped_field_op filled out - TX_grouped_field_op.csv')

# rename Database ID as Entity ID
ASKY_top_op = ASKY_top_op.rename(columns={'Database ID': 'Entity ID'})

# update based on top operator cut offs
ASKY_top_op_for_merge = ASKY_top_op[['operator_name', 'Entity ID']].copy()

# merge Entity ID based on operator_name
ASKY_Form_ID = ASKY_df_grouped_formation.merge(ASKY_top_op_for_merge, on='operator_name', how='left')
ASKY_Field_ID = ASKY_df_grouped_field.merge(ASKY_top_op_for_merge, on='operator_name', how='left')

# drop rows where Entity ID is null
ASKY_Form_ID_noNull = ASKY_Form_ID.dropna(subset=['Entity ID']).copy()
ASKY_Field_ID_noNull = ASKY_Field_ID.dropna(subset=['Entity ID']).copy()

# print row count pre and post drop formation and field
print(f'Formation grouped has {len(ASKY_df_grouped_formation)} rows')
print(f'Formation groups with only top ops has {len(ASKY_Form_ID_noNull)} rows')
print(f'Field grouped has {len(ASKY_df_grouped_field)} rows')
print(f'Field groups with only top ops has {len(ASKY_Field_ID_noNull)} rows')


Formation grouped has 16561 rows
Formation groups with only top ops has 1782 rows
Field grouped has 22078 rows
Field groups with only top ops has 2960 rows


In [ ]:
# check totals
ASKY_dry_gas_vol_sum_form_top_op = ASKY_Form_ID_noNull['dry_gas_vol'].sum()
ASKY_gross_gas_vol_sum_form_top_op = ASKY_Form_ID_noNull['gross_gas_vol'].sum()
ASKY_oil_and_condensate_vol_sum_form_top_op = ASKY_Form_ID_noNull['oil_and_condensate_vol'].sum()
ASKY_ngl_vol_sum_form_top_op = ASKY_Form_ID_noNull['ngl_vol'].sum()

ASKY_dry_gas_vol_sum_field_top_op = ASKY_Field_ID_noNull['dry_gas_vol'].sum()
ASKY_gross_gas_vol_sum_field_top_op = ASKY_Field_ID_noNull['gross_gas_vol'].sum()
ASKY_oil_and_condensate_vol_sum_field_top_op = ASKY_Field_ID_noNull['oil_and_condensate_vol'].sum()
ASKY_ngl_vol_sum_field_top_op = ASKY_Field_ID_noNull['ngl_vol'].sum()

print(f'Top op percent of formation grouped dry gas = {ASKY_dry_gas_vol_sum_form_top_op / ASKY_dry_gas_vol_sum_form:.2%}')
print(f'Top op percent of formation grouped gross gas sum = {ASKY_gross_gas_vol_sum_form_top_op / ASKY_gross_gas_vol_sum_form:.2%}')
print(f'Top op percent of formation grouped oil and condensate sum = {ASKY_oil_and_condensate_vol_sum_form_top_op / ASKY_oil_and_condensate_vol_sum_form:.2%}')
print(f'Top op percent of formation grouped ngl sum = {ASKY_ngl_vol_sum_form_top_op / ASKY_ngl_vol_sum_form:.2%}')

print(f'Top op percent of field grouped dry gas = {ASKY_dry_gas_vol_sum_field_top_op / ASKY_dry_gas_vol_sum_field:.2%}')
print(f'Top op percent of field grouped gross gas sum = {ASKY_gross_gas_vol_sum_field_top_op / ASKY_gross_gas_vol_sum_field:.2%}')
print(f'Top op percent of field grouped oil and condensate sum = {ASKY_oil_and_condensate_vol_sum_field_top_op / ASKY_oil_and_condensate_vol_sum_field:.2%}')
print(f'Top op percent of field grouped ngl sum = {ASKY_ngl_vol_sum_field_top_op / ASKY_ngl_vol_sum_field:.2%}')


Top op percent of formation grouped dry gas = 89.61%
Top op percent of formation grouped gross gas sum = 90.24%
Top op percent of formation grouped oil and condensate sum = 90.49%
Top op percent of formation grouped ngl sum = 94.34%
Top op percent of field grouped dry gas = 89.61%
Top op percent of field grouped gross gas sum = 90.24%
Top op percent of field grouped oil and condensate sum = 90.49%
Top op percent of field grouped ngl sum = 94.34%


In [ ]:
# adding in previously written texas code from https://colab.research.google.com/drive/1m_UD1GSsmFjzyObIlEgAHJSlLM3wrwU9?usp=sharing

# Pennsylvania
2024 data had 1,240 million boe of production across 41 assets

In [ ]:
# set year and state
key_year = 2024
active_state = 'pennsylvania'
activestatekeyyear = df[(df['state'] == active_state) & (df['year'] == key_year)]
print(f"The 'df' DataFrame has been filtered for the state of {active_state} and the year {key_year}, resulting in {len(activestatekeyyear)} rows.")

The 'df' DataFrame has been filtered for the state of pennsylvania and the year 2024, resulting in 86110 rows.


In [ ]:
# clean names and fix capitalization
ASKY_df_capitalized = activestatekeyyear.copy()

ASKY_df_capitalized['operator_name'] = ASKY_df_capitalized['operator_name'].str.title()
ASKY_df_capitalized['field_name'] = ASKY_df_capitalized['field_name'].str.title()
ASKY_df_capitalized['formation_name'] = ASKY_df_capitalized['formation_name'].str.title()

ASKY_df_capitalized['longitude'] = pd.to_numeric(ASKY_df_capitalized['longitude'], errors='coerce')
ASKY_df_capitalized['latitude'] = pd.to_numeric(ASKY_df_capitalized['latitude'], errors='coerce')

ASKY_df_capitalized['formation_name'] = ASKY_df_capitalized['formation_name'].fillna('Formation not assigned')
ASKY_df_capitalized['field_name'] = ASKY_df_capitalized['field_name'].fillna('Field not assigned')

ASKY_df_capitalized['field_name'] = ASKY_df_capitalized['field_name'].str.replace('Wc', 'Wild cat', regex=False)

ASKY_df_capitalized['operator_name'] = ASKY_df_capitalized['operator_name'].fillna('Operator not assigned')

print(f"{active_state} data has been cleaned and standardized.")

pennsylvania data has been cleaned and standardized.


In [ ]:
# group by field
ASKY_df_grouped_field = ASKY_df_capitalized.groupby(['field_name', 'operator_name']).agg({
    'well_no': lambda x: '; '.join(x.astype(str).unique()),
    'longitude': 'mean',
    'latitude': 'mean',
    'gross_gas_vol': 'sum',
    'dry_gas_vol': 'sum',
    'oil_and_condensate_vol': 'sum',
    'ngl_vol': 'sum',
    'formation_name': lambda x: '; '.join(x.astype(str).unique()),
    'resource_type': lambda x: '; '.join(x.astype(str).unique())
}).reset_index()

print(f'{active_state} has {len(ASKY_df_grouped_field)} rows when grouped by field + operator')
print(f'{active_state} has {len(ASKY_df_grouped_field[ASKY_df_grouped_field["field_name"] == "Field not assigned"])} rows with "Field not assigned"')

pennsylvania has 4588 rows when grouped by field + operator
pennsylvania has 182 rows with "Field not assigned"


In [ ]:
# group by formation
ASKY_df_grouped_formation = ASKY_df_capitalized.groupby(['formation_name', 'operator_name']).agg({
    'well_no': lambda x: '; '.join(x.astype(str).unique()),
    'longitude': 'mean',
    'latitude': 'mean',
    'gross_gas_vol': 'sum',
    'dry_gas_vol': 'sum',
    'oil_and_condensate_vol': 'sum',
    'ngl_vol': 'sum',
    'resource_type': lambda x: '; '.join(x.astype(str).unique())
}).reset_index()

print(f'{active_state} has {len(ASKY_df_grouped_formation)} rows when grouped by formation + operator')
print(f'{active_state} has {len(ASKY_df_grouped_formation[ASKY_df_grouped_formation["formation_name"] == "Formation not assigned"])} rows with "Formation not assigned"')

pennsylvania has 1002 rows when grouped by formation + operator
pennsylvania has 751 rows with "Formation not assigned"


In [ ]:
# check totals
ASKY_dry_gas_vol_sum_form = ASKY_df_grouped_formation['dry_gas_vol'].sum()
ASKY_gross_gas_vol_sum_form = ASKY_df_grouped_formation['gross_gas_vol'].sum()
ASKY_oil_and_condensate_vol_sum_form = ASKY_df_grouped_formation['oil_and_condensate_vol'].sum()
ASKY_ngl_vol_sum_form = ASKY_df_grouped_formation['ngl_vol'].sum()

ASKY_dry_gas_vol_sum_field = ASKY_df_grouped_field['dry_gas_vol'].sum()
ASKY_gross_gas_vol_sum_field = ASKY_df_grouped_field['gross_gas_vol'].sum()
ASKY_oil_and_condensate_vol_sum_field = ASKY_df_grouped_field['oil_and_condensate_vol'].sum()
ASKY_ngl_vol_sum_field = ASKY_df_grouped_field['ngl_vol'].sum()

# check if sums are equal in both formation and field
if ASKY_dry_gas_vol_sum_form == ASKY_dry_gas_vol_sum_field and ASKY_gross_gas_vol_sum_form == ASKY_gross_gas_vol_sum_field and ASKY_oil_and_condensate_vol_sum_form == ASKY_oil_and_condensate_vol_sum_field and ASKY_ngl_vol_sum_form == ASKY_ngl_vol_sum_field: print('sums match')
else: print('sums do not match')

print(f'Formation grouped dry gas sum = {ASKY_dry_gas_vol_sum_form}')
print(f'Formation grouped gross gas sum = {ASKY_gross_gas_vol_sum_form}')
print(f'Formation grouped oil and condensate sum = {ASKY_oil_and_condensate_vol_sum_form}')
print(f'Formation grouped ngl sum = {ASKY_ngl_vol_sum_form}')

print(f'Field grouped dry gas sum = {ASKY_dry_gas_vol_sum_field}')
print(f'Field grouped gross gas sum = {ASKY_gross_gas_vol_sum_field}')
print(f'Field grouped oil and condensate sum = {ASKY_oil_and_condensate_vol_sum_field}')
print(f'Field grouped ngl sum = {ASKY_ngl_vol_sum_field}')

sums match
Formation grouped dry gas sum = 207583.81471749526
Formation grouped gross gas sum = 211087.17269898666
Formation grouped oil and condensate sum = 4.46097186
Formation grouped ngl sum = 85.21173549750708
Field grouped dry gas sum = 207583.81471749526
Field grouped gross gas sum = 211087.17269898666
Field grouped oil and condensate sum = 4.46097186
Field grouped ngl sum = 85.21173549750708


In [ ]:
# parse operators, to add in Entity ID's manually
# calculate total production
ASKY_df_grouped_field['total_hydrocarbons_production'] = (ASKY_df_grouped_field['dry_gas_vol']*0.005883) + ASKY_df_grouped_field['oil_and_condensate_vol']+ ASKY_df_grouped_field['ngl_vol']
ASKY_df_grouped_formation['total_hydrocarbons_production'] =  (ASKY_df_grouped_formation['dry_gas_vol']*0.005883) + ASKY_df_grouped_formation['oil_and_condensate_vol'] + ASKY_df_grouped_formation['ngl_vol']

# group by operator for datacleaning
ASKY_df_grouped_formation_op = ASKY_df_grouped_formation.groupby('operator_name').agg({'total_hydrocarbons_production': 'sum'})

# sort by production high to low
ASKY_df_grouped_formation_op =ASKY_df_grouped_formation_op.sort_values(by='total_hydrocarbons_production', ascending=False)

PA_GOGET_Folder = '/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/Data sets & reports (O&G)/O&G country materials/United States (GOGET O&G)/Pennsylvania (GOGET)'
ASKY_df_grouped_formation_op.to_csv(f'{PA_GOGET_Folder}/PA_grouped_field_op.csv')


In [ ]:
# load in operator data (manually gathered)
ASKY_top_op = pd.read_csv(f'{PA_GOGET_Folder}/PA_grouped_field_op -  filled out - PA_grouped_field_op.csv')

# rename Database ID as Entity ID
ASKY_top_op = ASKY_top_op.rename(columns={'Database ID': 'Entity ID'})

# update based on top operator cut offs
ASKY_top_op_for_merge = ASKY_top_op[['operator_name', 'Entity ID']].copy()

# merge Entity ID based on operator_name
ASKY_Form_ID = ASKY_df_grouped_formation.merge(ASKY_top_op_for_merge, on='operator_name', how='left')
ASKY_Field_ID = ASKY_df_grouped_field.merge(ASKY_top_op_for_merge, on='operator_name', how='left')

# drop rows where Entity ID is null
ASKY_Form_ID_noNull = ASKY_Form_ID.dropna(subset=['Entity ID']).copy()
ASKY_Field_ID_noNull = ASKY_Field_ID.dropna(subset=['Entity ID']).copy()

# print row count pre and post drop formation and field
print(f'Formation grouped has {len(ASKY_df_grouped_formation)} rows')
print(f'Formation groups with only top ops has {len(ASKY_Form_ID_noNull)} rows')
print(f'Field grouped has {len(ASKY_df_grouped_field)} rows')
print(f'Field groups with only top ops has {len(ASKY_Field_ID_noNull)} rows')

Formation grouped has 1002 rows
Formation groups with only top ops has 218 rows
Field grouped has 4588 rows
Field groups with only top ops has 1007 rows


In [ ]:
# check totals
ASKY_dry_gas_vol_sum_form_top_op = ASKY_Form_ID_noNull['dry_gas_vol'].sum()
ASKY_gross_gas_vol_sum_form_top_op = ASKY_Form_ID_noNull['gross_gas_vol'].sum()
ASKY_oil_and_condensate_vol_sum_form_top_op = ASKY_Form_ID_noNull['oil_and_condensate_vol'].sum()
ASKY_ngl_vol_sum_form_top_op = ASKY_Form_ID_noNull['ngl_vol'].sum()

ASKY_dry_gas_vol_sum_field_top_op = ASKY_Field_ID_noNull['dry_gas_vol'].sum()
ASKY_gross_gas_vol_sum_field_top_op = ASKY_Field_ID_noNull['gross_gas_vol'].sum()
ASKY_oil_and_condensate_vol_sum_field_top_op = ASKY_Field_ID_noNull['oil_and_condensate_vol'].sum()
ASKY_ngl_vol_sum_field_top_op = ASKY_Field_ID_noNull['ngl_vol'].sum()

print(f'Top op percent of formation grouped dry gas = {ASKY_dry_gas_vol_sum_form_top_op / ASKY_dry_gas_vol_sum_form:.2%}')
print(f'Top op percent of formation grouped gross gas sum = {ASKY_gross_gas_vol_sum_form_top_op / ASKY_gross_gas_vol_sum_form:.2%}')
print(f'Top op percent of formation grouped oil and condensate sum = {ASKY_oil_and_condensate_vol_sum_form_top_op / ASKY_oil_and_condensate_vol_sum_form:.2%}')
print(f'Top op percent of formation grouped ngl sum = {ASKY_ngl_vol_sum_form_top_op / ASKY_ngl_vol_sum_form:.2%}')

print(f'Top op percent of field grouped dry gas = {ASKY_dry_gas_vol_sum_field_top_op / ASKY_dry_gas_vol_sum_field:.2%}')
print(f'Top op percent of field grouped gross gas sum = {ASKY_gross_gas_vol_sum_field_top_op / ASKY_gross_gas_vol_sum_field:.2%}')
print(f'Top op percent of field grouped oil and condensate sum = {ASKY_oil_and_condensate_vol_sum_field_top_op / ASKY_oil_and_condensate_vol_sum_field:.2%}')
print(f'Top op percent of field grouped ngl sum = {ASKY_ngl_vol_sum_field_top_op / ASKY_ngl_vol_sum_field:.2%}')

Top op percent of formation grouped dry gas = 99.15%
Top op percent of formation grouped gross gas sum = 99.15%
Top op percent of formation grouped oil and condensate sum = 83.04%
Top op percent of formation grouped ngl sum = 99.15%
Top op percent of field grouped dry gas = 99.15%
Top op percent of field grouped gross gas sum = 99.15%
Top op percent of field grouped oil and condensate sum = 83.04%
Top op percent of field grouped ngl sum = 99.15%


In [ ]:
# select field or formation
row_choice = 'formation' # User can change this to 'formation'

if row_choice == 'formation':
    choice = ASKY_Form_ID_noNull
    name_col = 'formation_name'
    project_complex_col = 'formation_name'
elif row_choice == 'field':
    choice = ASKY_Field_ID_noNull
    name_col = 'field_name'
    project_complex_col = 'formation_name'
else:
    raise ValueError("Invalid row_choice. Must be 'formation' or 'field'.")

print(f'Choice is {row_choice}')
choice.head()

Choice is formation


,formation_name,operator_name,well_no,longitude,latitude,gross_gas_vol,dry_gas_vol,oil_and_condensate_vol,ngl_vol,resource_type,total_hydrocarbons_production,Entity ID
0,Black River Limestone;Utica Shale,Jklm Energy Llc,105-21915,-77.927880,41.725960,20.548375,20.207339,0.000000,0.008295,Unconventional,0.127175,E100002021383
1,Bradford,Atlas Resources Llc,051-24005; 051-24011; 051-24014; 051-24018; 05...,-79.839468,39.951937,0.348867,0.343077,0.000011,0.000141,Conventional,0.002170,E100000131297
3,Bradford;Marcellus,Snyder Bros Inc,019-21822,-79.719525,40.796700,0.033980,0.033416,0.000000,0.000014,Unconventional,0.000210,E100000192180
5,Burket,Cnx Gas Co Llc,003-22303; 059-26508; 059-26509; 059-26667; 05...,-80.337728,40.070882,69.936279,68.775565,0.000000,0.028232,Unconventional,0.432839,E100000192173
6,Burket,Em Energy Pa Llc,019-22075; 019-22127; 019-22128; 019-22172; 01...,-79.851593,40.976481,71.890095,70.696954,0.004479,0.029021,Unconventional,0.449410,E100000192197


In [ ]:
# standardize the 'resource_type' column for GOGET bulk upload

choice['resource_type'] = choice['resource_type'].replace({
    'Unconventional; Conventional': 'Unconventional',
    'Conventional; Unconventional': 'Unconventional',
    'Conventional': 'Conventional'
})

print("Resource types standardized.")

Resource types standardized.


In [ ]:
# define fuel type
import numpy as np

conditions = [
    (choice['dry_gas_vol'] > 0) & (choice['oil_and_condensate_vol'] > 0),
    (choice['dry_gas_vol'] == 0) & (choice['oil_and_condensate_vol'] > 0),
    (choice['dry_gas_vol'] > 0) & (choice['oil_and_condensate_vol'] == 0)
]

choices = ['oil and gas', 'oil', 'gas']

choice['fuel_type'] = np.select(conditions, choices, default=None)

print("Fuel types have been assigned.")

Fuel types have been assigned.


In [ ]:
# format names based on field or formation list - add source
choice.loc[:, 'name'] = choice[name_col] + ' - ' + choice['operator_name']
choice.loc[:, 'unit_type'] = "asset"
choice.loc[:, 'project_complex'] = choice[project_complex_col]
choice.loc[:, 'coordinate_accuracy'] = "exact"
choice.loc[:, 'operator_source'] = "PAProduction2024"
choice.loc[:, 'location_source'] = "PAProduction2024"
choice.loc[:, 'resource_type_source'] = "PAProduction2024"
choice.loc[:, 'discovery_year_source'] = "PAProduction2024"
choice['name'] = choice['name'].str.replace('Wc', 'Wildcat', regex=False)

print("Data formatted for GOGET bulk upload.")

Data formatted for GOGET bulk upload.


In [ ]:
# add total production column if it doesn't already exist (it should from the earlier step, but this is a safeguard)
if 'total_hydrocarbons_production' not in choice.columns:
    choice['total_hydrocarbons_production'] = (choice['dry_gas_vol']*0.005883) + choice['oil_and_condensate_vol'] + choice['ngl_vol']

# Select and order the final columns for ASKY_Clean
ASKY_Clean = choice[[
    'name',
    'unit_type',
    'fuel_type',
    'project_complex',
    'longitude',
    'latitude',
    'coordinate_accuracy',
    'location_source',
    'operator_name',
    'operator_source',
    'Entity ID',
    'resource_type',
    'resource_type_source',
    # 'Discovery year',
    # 'discovery_year_source',
    'well_no',
    'total_hydrocarbons_production'
]].copy()

# sort by total_hydrocarbons_production high to low
ASKY_Clean = ASKY_Clean.sort_values(by='total_hydrocarbons_production', ascending=False).reset_index(drop=True)

print("ASKY_Clean DataFrame created with selected and ordered columns.")
display(ASKY_Clean.head())

ASKY_Clean DataFrame created with selected and ordered columns.


,name,unit_type,fuel_type,project_complex,longitude,latitude,coordinate_accuracy,location_source,operator_name,operator_source,Entity ID,resource_type,resource_type_source,well_no,total_hydrocarbons_production
0,Marcellus - Chesapeake Appalachia Llc,asset,gas,Marcellus,-76.296538,41.676397,exact,PAProduction2024,Chesapeake Appalachia Llc,PAProduction2024,E100000192168,Unconventional,PAProduction2024,007-20315; 015-20085; 015-20097; 015-20113; 01...,185.188319
1,Marcellus - Coterra Energy Inc,asset,gas,Marcellus,-75.834950,41.750745,exact,PAProduction2024,Coterra Energy Inc,PAProduction2024,E100001014513,Unconventional,PAProduction2024,115-20007; 115-20008; 115-20010; 115-20011; 11...,151.021627
2,Marcellus - Range Resources Appalachia Llc,asset,oil and gas,Marcellus,-79.914951,40.396125,exact,PAProduction2024,Range Resources Appalachia Llc,PAProduction2024,E100002016338,Unconventional,PAProduction2024,003-21982; 003-22193; 003-22194; 003-22195; 00...,130.588437
3,Marcellus - Eqt Prod Co,asset,oil and gas,Marcellus,-79.958503,40.138866,exact,PAProduction2024,Eqt Prod Co,PAProduction2024,E100000191299,Unconventional,PAProduction2024,003-22234; 003-22235; 003-22236; 003-22237; 00...,108.039322
4,Marcellus - Rice Drilling B Llc,asset,gas,Marcellus,-80.212887,39.974623,exact,PAProduction2024,Rice Drilling B Llc,PAProduction2024,E100000192171,Unconventional,PAProduction2024,059-25446; 059-25467; 059-25468; 059-25469; 05...,68.824674


In [ ]:
# check totals in choice
ASKY_choice_total = ASKY_Clean['total_hydrocarbons_production'].sum()


# print totals
print(f'{active_state} parsed data has {ASKY_choice_total} million boe')

pennsylvania parsed data has 1299.0247902269361 million boe


In [ ]:
# save out ASKY_Clean for manual matching
ASKY_Clean.to_excel(f'{PA_GOGET_Folder}/PA_GOGET_Clean_for_ID_Match.xlsx', index=False)

In [ ]:
GOGET_IDs = pd.read_csv(f'{PA_GOGET_Folder}/PA_GOGET_Clean_for_ID_Match.xlsx - Manually Matched - fixed - PA_GOGET_Clean_for_ID_Match.xlsx - Manually Matched.csv') # fuzzy match was not owrking, so i manually matched these

# add goget IDs matching on name to append "GOGET_Unit_ID"
ASKY_Clean = ASKY_Clean.merge(GOGET_IDs[['name', 'GOGET_Unit_ID']], on='name', how='left')

ASKY_Clean

,name,unit_type,fuel_type,project_complex,longitude,latitude,coordinate_accuracy,location_source,operator_name,operator_source,Entity ID,resource_type,resource_type_source,well_no,total_hydrocarbons_production,GOGET_Unit_ID_x,GOGET_Unit_ID_y
0,Marcellus - Chesapeake Appalachia Llc,asset,gas,Marcellus,-76.296538,41.676397,exact,PAProduction2024,Chesapeake Appalachia Llc,PAProduction2024,E100000192168,Unconventional,PAProduction2024,007-20315; 015-20085; 015-20097; 015-20113; 01...,185.188319,L100000322548,L100000322548
1,Marcellus - Coterra Energy Inc,asset,gas,Marcellus,-75.834950,41.750745,exact,PAProduction2024,Coterra Energy Inc,PAProduction2024,E100001014513,Unconventional,PAProduction2024,115-20007; 115-20008; 115-20010; 115-20011; 11...,151.021627,L100000322549,L100000322549
2,Marcellus - Range Resources Appalachia Llc,asset,oil and gas,Marcellus,-79.914951,40.396125,exact,PAProduction2024,Range Resources Appalachia Llc,PAProduction2024,E100002016338,Unconventional,PAProduction2024,003-21982; 003-22193; 003-22194; 003-22195; 00...,130.588437,L100000322550,L100000322550
3,Marcellus - Eqt Prod Co,asset,oil and gas,Marcellus,-79.958503,40.138866,exact,PAProduction2024,Eqt Prod Co,PAProduction2024,E100000191299,Unconventional,PAProduction2024,003-22234; 003-22235; 003-22236; 003-22237; 00...,108.039322,L100000322551,L100000322551
4,Marcellus - Rice Drilling B Llc,asset,gas,Marcellus,-80.212887,39.974623,exact,PAProduction2024,Rice Drilling B Llc,PAProduction2024,E100000192171,Unconventional,PAProduction2024,059-25446; 059-25467; 059-25468; 059-25469; 05...,68.824674,L100000322552,L100000322552
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
213,Marcellus;Mandata Shale - Coterra Energy Inc,asset,gas,Marcellus;Mandata Shale,-75.944950,41.864400,exact,PAProduction2024,Coterra Energy Inc,PAProduction2024,E100001014513,Unconventional,PAProduction2024,115-20447,0.000057,NaN,NaN
214,Formation not assigned - Em Energy Pa Llc,asset,gas,Formation not assigned,-79.855259,40.949001,exact,PAProduction2024,Em Energy Pa Llc,PAProduction2024,E100000192197,Conventional,PAProduction2024,019-21443,0.000037,NaN,NaN
215,Marcellus;Huntersville Chert - Snyder Bros Inc,asset,gas,Marcellus;Huntersville Chert,-79.610317,40.718103,exact,PAProduction2024,Snyder Bros Inc,PAProduction2024,E100000192180,Unconventional,PAProduction2024,005-30792,0.000028,NaN,NaN
216,Venango - Atlas Resources Llc,asset,gas,Venango,-79.822357,39.729837,exact,PAProduction2024,Atlas Resources Llc,PAProduction2024,E100000131297,Conventional,PAProduction2024,051-24025,0.000017,NaN,NaN


In [ ]:
production_cols = ['gross_gas_vol', 'dry_gas_vol', 'oil_and_condensate_vol', 'ngl_vol']
id_cols = [col for col in choice.columns if col not in production_cols]

ASKY_long_prod = pd.melt(choice,
                                  id_vars=id_cols,
                                  value_vars=production_cols,
                                  var_name='production_type',
                                  value_name='volume')

cols_to_drop_from_melted = ['Entity ID','well_no','longitude','latitude','unit_type', 'project_complex','coordinate_accuracy','operator_source','location_source', 'field_name','resource_type','resource_type_source','fuel_type', 'operator_name','discovery_year_source' ,'total_hydrocarbons_production']

# Dynamically add 'formation_name' if it was not used as name_col
if row_choice == 'field': # if row_choice is 'field', formation_name is still a column and needs to be dropped
    if 'formation_name' in ASKY_long_prod.columns:
        cols_to_drop_from_melted.append('formation_name')

existing_cols_to_drop = [col for col in cols_to_drop_from_melted if col in ASKY_long_prod.columns]
ASKY_long_prod.drop(columns=existing_cols_to_drop, inplace=True)

print(f'{active_state} parsed data has been melted into long format')
display(ASKY_long_prod.head())

pennsylvania parsed data has been melted into long format


,formation_name,name,production_type,volume
0,Black River Limestone;Utica Shale,Black River Limestone;Utica Shale - Jklm Energ...,gross_gas_vol,20.548375
1,Bradford,Bradford - Atlas Resources Llc,gross_gas_vol,0.348867
2,Bradford;Marcellus,Bradford;Marcellus - Snyder Bros Inc,gross_gas_vol,0.033980
3,Burket,Burket - Cnx Gas Co Llc,gross_gas_vol,69.936279
4,Burket,Burket - Em Energy Pa Llc,gross_gas_vol,71.890095


In [ ]:
ASKY_long_prod['fuel description'] = ASKY_long_prod['production_type'].replace({'gross_gas_vol': 'gross gas', 'dry_gas_vol': 'dry gas', 'oil_and_condensate_vol': 'oil and condensates', 'ngl_vol': 'NGL'})

ASKY_long_prod['Year'] = key_year

ASKY_long_prod['production/reserves'] = 'production'

ASKY_long_prod.rename(columns={'volume': 'Quantity'}, inplace=True)

ASKY_long_prod['quantity_unit'] = ASKY_long_prod['fuel description'].apply(lambda x: 'million cubic meters/y' if x in ['gross gas', 'dry gas'] else 'million bbl/y')

ASKY_long_prod['Datasource'] = 'PAProduction2024'

print("ASKY_long_prod has been formatted for GOGET bulk upload.")
display(ASKY_long_prod.head())

ASKY_long_prod has been formatted for GOGET bulk upload.


,formation_name,name,production_type,Quantity,fuel description,Year,production/reserves,quantity_unit,Datasource
0,Black River Limestone;Utica Shale,Black River Limestone;Utica Shale - Jklm Energ...,gross_gas_vol,20.548375,gross gas,2024,production,million cubic meters/y,PAProduction2024
1,Bradford,Bradford - Atlas Resources Llc,gross_gas_vol,0.348867,gross gas,2024,production,million cubic meters/y,PAProduction2024
2,Bradford;Marcellus,Bradford;Marcellus - Snyder Bros Inc,gross_gas_vol,0.033980,gross gas,2024,production,million cubic meters/y,PAProduction2024
3,Burket,Burket - Cnx Gas Co Llc,gross_gas_vol,69.936279,gross gas,2024,production,million cubic meters/y,PAProduction2024
4,Burket,Burket - Em Energy Pa Llc,gross_gas_vol,71.890095,gross gas,2024,production,million cubic meters/y,PAProduction2024


In [ ]:
import datetime
time_stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

ASKY_Clean.to_excel(f'{PA_GOGET_Folder}/{active_state}_AI_FOR_GOGET{key_year}_{time_stamp}.xlsx', index=False)
ASKY_long_prod.to_excel(f'{PA_GOGET_Folder}/{active_state}_grouped_formation_long_{key_year}.xlsx', index=False)

print(f'Saved {active_state}_AI_FOR_GOGET{key_year}_{time_stamp}.xlsx and {active_state}_grouped_formation_long_{key_year}.xlsx to {PA_GOGET_Folder}')

Saved pennsylvania_AI_FOR_GOGET2024_2026-01-02_23-02-25.xlsx and pennsylvania_grouped_formation_long_2024.xlsx to /content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/Data sets & reports (O&G)/O&G country materials/United States (GOGET O&G)/Pennsylvania (GOGET)


# New Mexico


In [ ]:
# set year and state
key_year = 2024
active_state = 'new_mexico'
activestatekeyyear = df[(df['state'] == active_state) & (df['year'] == key_year)]
print(f"The 'df' DataFrame has been filtered for the state of {active_state} and the year {key_year}, resulting in {len(activestatekeyyear)} rows.")

The 'df' DataFrame has been filtered for the state of new_mexico and the year 2024, resulting in 65990 rows.


In [ ]:
# add in formation parsed here https://colab.research.google.com/drive/152Mdo5hBz_PsVZzRHgRO34sJOwzR2SCl#scrollTo=XgIA2Avx2ek3
formation_file = '/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/Data sets & reports (O&G)/O&G country materials/United States (GOGET O&G)/New Mexico (GOGET)/County Level Data (GOGET)/Location and General Data 2025/api_formation_all_2026-01-15_18-06-42.csv'
formation_df = pd.read_csv(formation_file)
# Ensure the 'api' column in formation_df is of string type for merging
formation_df['api'] = formation_df['api'].astype(str)
formation_df.head()

,api,formation
0,3000305000,NaN
1,3000305001,NaN
2,3000305002,NaN
3,3000305007,NaN
4,3000305012,NaN


In [ ]:
activestatekeyyear.columns

Index(['country', 'state', 'county_name', 'year', 'well_no', 'well_name',
       'field_no', 'field_name', 'lease_no', 'lease_name', 'formation_no',
       'formation_name', 'longitude', 'latitude', 'operator_no',
       'operator_name', 'owner_no', 'owner_name', 'ownership_perc',
       'resource_type', 'resource_type_detail', 'recovery_mechanism',
       'field_terrain', 'oil_gravity', 'gas_gravity', 'oil_and_condensate_vol',
       'gross_gas_vol', 'dry_gas_vol', 'ngl_vol'],
      dtype='object')

In [ ]:
# remove dashes in well_no for matching
activestatekeyyear['api'] = activestatekeyyear['well_no'].str.replace('-', '')

/tmp/ipython-input-309195208.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  activestatekeyyear['api'] = activestatekeyyear['well_no'].str.replace('-', '')


In [ ]:
# matching well_no and api, append formation as formation_gotech
activestatekeyyear = activestatekeyyear.merge(formation_df[['api', 'formation']], left_on='api', right_on='api', how='left')
activestatekeyyear.head()

,country,state,county_name,year,well_no,well_name,field_no,field_name,lease_no,lease_name,...,recovery_mechanism,field_terrain,oil_gravity,gas_gravity,oil_and_condensate_vol,gross_gas_vol,dry_gas_vol,ngl_vol,api,formation
0,united_states,new_mexico,None,2024.0,30-005-00130,HARRIS #001,None,None,None,None,...,None,Onshore,NaN,NaN,0.000798,0.0,0.0,0.0,3000500130,San Andres
1,united_states,new_mexico,None,2024.0,30-005-00134,STATE E 92 #001,None,None,None,None,...,None,Onshore,NaN,NaN,0.000039,0.0,0.0,0.0,3000500134,Queen-Grayburg
2,united_states,new_mexico,None,2024.0,30-005-00135,CHAVES P STATE #002,None,None,None,None,...,None,Onshore,NaN,NaN,0.000149,0.0,0.0,0.0,3000500135,Queen-Grayburg
3,united_states,new_mexico,None,2024.0,30-005-00137,KELLY A STATE #001,None,None,None,None,...,None,Onshore,NaN,NaN,0.000142,0.0,0.0,0.0,3000500137,Queen-Grayburg
4,united_states,new_mexico,None,2024.0,30-005-00141,STATE E 92 #003,None,None,None,None,...,None,Onshore,NaN,NaN,0.000019,0.0,0.0,0.0,3000500141,Queen-Grayburg


In [ ]:
# rename "formation" as "NM_added_formation"
activestatekeyyear.rename(columns={'formation': 'NM_added_formation'}, inplace=True)

In [ ]:
activestatekeyyear.columns

Index(['country', 'state', 'county_name', 'year', 'well_no', 'well_name',
       'field_no', 'field_name', 'lease_no', 'lease_name', 'formation_no',
       'formation_name', 'longitude', 'latitude', 'operator_no',
       'operator_name', 'owner_no', 'owner_name', 'ownership_perc',
       'resource_type', 'resource_type_detail', 'recovery_mechanism',
       'field_terrain', 'oil_gravity', 'gas_gravity', 'oil_and_condensate_vol',
       'gross_gas_vol', 'dry_gas_vol', 'ngl_vol', 'api', 'NM_added_formation'],
      dtype='object')

In [ ]:
# drop existing formation_name
activestatekeyyear.drop(columns=['formation_name'], inplace=True)

In [ ]:
# clean names and fix capitalization
ASKY_df_capitalized = activestatekeyyear.copy()

ASKY_df_capitalized['operator_name'] = ASKY_df_capitalized['operator_name'].str.title()
ASKY_df_capitalized['field_name'] = ASKY_df_capitalized['field_name'].str.title()
ASKY_df_capitalized['formation_name'] = ASKY_df_capitalized['NM_added_formation'].str.title()

# clean blank space on formation name
ASKY_df_capitalized['formation_name'] = ASKY_df_capitalized['formation_name'].str.strip()

ASKY_df_capitalized['longitude'] = pd.to_numeric(ASKY_df_capitalized['longitude'], errors='coerce')
ASKY_df_capitalized['latitude'] = pd.to_numeric(ASKY_df_capitalized['latitude'], errors='coerce')

ASKY_df_capitalized['formation_name'] = ASKY_df_capitalized['formation_name'].fillna('Formation not assigned')
ASKY_df_capitalized['field_name'] = ASKY_df_capitalized['field_name'].fillna('Field not assigned')

ASKY_df_capitalized['field_name'] = ASKY_df_capitalized['field_name'].str.replace('Wc', 'Wild cat', regex=False)

ASKY_df_capitalized['operator_name'] = ASKY_df_capitalized['operator_name'].fillna('Operator not assigned')

print(f"{active_state} data has been cleaned and standardized.")

new_mexico data has been cleaned and standardized.


In [ ]:
# sort ASKY_df_capitalized by oil_and_condensate_vol high to low
ASKY_df_capitalized = ASKY_df_capitalized.sort_values(by='oil_and_condensate_vol', ascending=False)

# save out wells to examine
NM_GOGET_Folder = '/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/Data sets & reports (O&G)/O&G country materials/United States (GOGET O&G)/New Mexico (GOGET)'
ASKY_df_capitalized.to_csv(f'{NM_GOGET_Folder}/NM_df_capitalized.csv', index=False)

In [ ]:
# group by field
ASKY_df_grouped_field = ASKY_df_capitalized.groupby(['field_name', 'operator_name']).agg({
    'well_no': lambda x: '; '.join(x.astype(str).unique()),
    'longitude': 'mean',
    'latitude': 'mean',
    'gross_gas_vol': 'sum',
    'dry_gas_vol': 'sum',
    'oil_and_condensate_vol': 'sum',
    'ngl_vol': 'sum',
    'formation_name': lambda x: '; '.join(x.astype(str).unique()),
    'resource_type': lambda x: '; '.join(x.astype(str).unique())
}).reset_index()

print(f'{active_state} has {len(ASKY_df_grouped_field)} rows when grouped by field + operator')
print(f'{active_state} has {len(ASKY_df_grouped_field[ASKY_df_grouped_field["field_name"] == "Field not assigned"])} rows with "Field not assigned"')

new_mexico has 386 rows when grouped by field + operator
new_mexico has 386 rows with "Field not assigned"


In [ ]:
# group by formation
ASKY_df_grouped_formation = ASKY_df_capitalized.groupby(['formation_name', 'operator_name']).agg({
    'well_no': lambda x: '; '.join(x.astype(str).unique()),
    'longitude': 'mean',
    'latitude': 'mean',
    'gross_gas_vol': 'sum',
    'dry_gas_vol': 'sum',
    'oil_and_condensate_vol': 'sum',
    'ngl_vol': 'sum',
    'resource_type': lambda x: '; '.join(x.astype(str).unique())
}).reset_index()

print(f'{active_state} has {len(ASKY_df_grouped_formation)} rows when grouped by formation + operator')
print(f'{active_state} has {len(ASKY_df_grouped_formation[ASKY_df_grouped_formation["formation_name"] == "Formation not assigned"])} rows with "Formation not assigned"')

new_mexico has 2894 rows when grouped by formation + operator
new_mexico has 265 rows with "Formation not assigned"


In [ ]:
# check totals
ASKY_dry_gas_vol_sum_form = ASKY_df_grouped_formation['dry_gas_vol'].sum()
ASKY_gross_gas_vol_sum_form = ASKY_df_grouped_formation['gross_gas_vol'].sum()
ASKY_oil_and_condensate_vol_sum_form = ASKY_df_grouped_formation['oil_and_condensate_vol'].sum()
ASKY_ngl_vol_sum_form = ASKY_df_grouped_formation['ngl_vol'].sum()

ASKY_dry_gas_vol_sum_field = ASKY_df_grouped_field['dry_gas_vol'].sum()
ASKY_gross_gas_vol_sum_field = ASKY_df_grouped_field['gross_gas_vol'].sum()
ASKY_oil_and_condensate_vol_sum_field = ASKY_df_grouped_field['oil_and_condensate_vol'].sum()
ASKY_ngl_vol_sum_field = ASKY_df_grouped_field['ngl_vol'].sum()

# check if sums are equal in both formation and field
if ASKY_dry_gas_vol_sum_form == ASKY_dry_gas_vol_sum_field and ASKY_gross_gas_vol_sum_form == ASKY_gross_gas_vol_sum_field and ASKY_oil_and_condensate_vol_sum_form == ASKY_oil_and_condensate_vol_sum_field and ASKY_ngl_vol_sum_form == ASKY_ngl_vol_sum_field: print('sums match')
else: print('sums do not match')

print(f'Formation grouped dry gas sum = {ASKY_dry_gas_vol_sum_form}')
print(f'Formation grouped gross gas sum = {ASKY_gross_gas_vol_sum_form}')
print(f'Formation grouped oil and condensate sum = {ASKY_oil_and_condensate_vol_sum_form}')
print(f'Formation grouped ngl sum = {ASKY_ngl_vol_sum_form}')

print(f'Field grouped dry gas sum = {ASKY_dry_gas_vol_sum_field}')
print(f'Field grouped gross gas sum = {ASKY_gross_gas_vol_sum_field}')
print(f'Field grouped oil and condensate sum = {ASKY_oil_and_condensate_vol_sum_field}')
print(f'Field grouped ngl sum = {ASKY_ngl_vol_sum_field}')

sums do not match
Formation grouped dry gas sum = 91954.41333264785
Formation grouped gross gas sum = 101316.9536960031
Formation grouped oil and condensate sum = 739.031557
Formation grouped ngl sum = 212.00064465719734
Field grouped dry gas sum = 91954.41333264785
Field grouped gross gas sum = 101316.95369600307
Field grouped oil and condensate sum = 739.031557
Field grouped ngl sum = 212.00064465719734


In [ ]:
# parse operators, to add in Entity ID's manually
# calculate total production
ASKY_df_grouped_field['total_hydrocarbons_production'] = (ASKY_df_grouped_field['dry_gas_vol']*0.005883) + ASKY_df_grouped_field['oil_and_condensate_vol']+ ASKY_df_grouped_field['ngl_vol']
ASKY_df_grouped_formation['total_hydrocarbons_production'] =  (ASKY_df_grouped_formation['dry_gas_vol']*0.005883) + ASKY_df_grouped_formation['oil_and_condensate_vol'] + ASKY_df_grouped_formation['ngl_vol']

# group by operator for datacleaning
ASKY_df_grouped_formation_op = ASKY_df_grouped_formation.groupby('operator_name').agg({'total_hydrocarbons_production': 'sum'})

# sort by production high to low
ASKY_df_grouped_formation_op =ASKY_df_grouped_formation_op.sort_values(by='total_hydrocarbons_production', ascending=False)

ASKY_df_grouped_formation_op.to_csv(f'{NM_GOGET_Folder}/NM_grouped_formation_op.csv')


In [ ]:
# load in operator data (manually gathered)
ASKY_top_op = pd.read_csv(f'{NM_GOGET_Folder}/NM_grouped_formation_op - NM Matched 2025 01 15.csv')

# rename Database ID as Entity ID
ASKY_top_op = ASKY_top_op.rename(columns={'Database ID': 'Entity ID'})

# update based on top operator cut offs
ASKY_top_op_for_merge = ASKY_top_op[['operator_name', 'Entity ID']].copy()

# merge Entity ID based on operator_name
ASKY_Form_ID = ASKY_df_grouped_formation.merge(ASKY_top_op_for_merge, on='operator_name', how='left')
ASKY_Field_ID = ASKY_df_grouped_field.merge(ASKY_top_op_for_merge, on='operator_name', how='left')

# drop rows where Entity ID is null
ASKY_Form_ID_noNull = ASKY_Form_ID.dropna(subset=['Entity ID']).copy()
ASKY_Field_ID_noNull = ASKY_Field_ID.dropna(subset=['Entity ID']).copy()

# print row count pre and post drop formation and field
print(f'Formation grouped has {len(ASKY_df_grouped_formation)} rows')
print(f'Formation groups with only top ops has {len(ASKY_Form_ID_noNull)} rows')
print(f'Field grouped has {len(ASKY_df_grouped_field)} rows')
print(f'Field groups with only top ops has {len(ASKY_Field_ID_noNull)} rows')

Formation grouped has 2894 rows
Formation groups with only top ops has 1243 rows
Field grouped has 386 rows
Field groups with only top ops has 54 rows


In [ ]:
# check totals
ASKY_dry_gas_vol_sum_form_top_op = ASKY_Form_ID_noNull['dry_gas_vol'].sum()
ASKY_gross_gas_vol_sum_form_top_op = ASKY_Form_ID_noNull['gross_gas_vol'].sum()
ASKY_oil_and_condensate_vol_sum_form_top_op = ASKY_Form_ID_noNull['oil_and_condensate_vol'].sum()
ASKY_ngl_vol_sum_form_top_op = ASKY_Form_ID_noNull['ngl_vol'].sum()

ASKY_dry_gas_vol_sum_field_top_op = ASKY_Field_ID_noNull['dry_gas_vol'].sum()
ASKY_gross_gas_vol_sum_field_top_op = ASKY_Field_ID_noNull['gross_gas_vol'].sum()
ASKY_oil_and_condensate_vol_sum_field_top_op = ASKY_Field_ID_noNull['oil_and_condensate_vol'].sum()
ASKY_ngl_vol_sum_field_top_op = ASKY_Field_ID_noNull['ngl_vol'].sum()

print(f'Top op percent of formation grouped dry gas = {ASKY_dry_gas_vol_sum_form_top_op / ASKY_dry_gas_vol_sum_form:.2%}')
print(f'Top op percent of formation grouped gross gas sum = {ASKY_gross_gas_vol_sum_form_top_op / ASKY_gross_gas_vol_sum_form:.2%}')
print(f'Top op percent of formation grouped oil and condensate sum = {ASKY_oil_and_condensate_vol_sum_form_top_op / ASKY_oil_and_condensate_vol_sum_form:.2%}')
print(f'Top op percent of formation grouped ngl sum = {ASKY_ngl_vol_sum_form_top_op / ASKY_ngl_vol_sum_form:.2%}')

print(f'Top op percent of field grouped dry gas = {ASKY_dry_gas_vol_sum_field_top_op / ASKY_dry_gas_vol_sum_field:.2%}')
print(f'Top op percent of field grouped gross gas sum = {ASKY_gross_gas_vol_sum_field_top_op / ASKY_gross_gas_vol_sum_field:.2%}')
print(f'Top op percent of field grouped oil and condensate sum = {ASKY_oil_and_condensate_vol_sum_field_top_op / ASKY_oil_and_condensate_vol_sum_field:.2%}')
print(f'Top op percent of field grouped ngl sum = {ASKY_ngl_vol_sum_field_top_op / ASKY_ngl_vol_sum_field:.2%}')

Top op percent of formation grouped dry gas = 98.63%
Top op percent of formation grouped gross gas sum = 98.63%
Top op percent of formation grouped oil and condensate sum = 98.94%
Top op percent of formation grouped ngl sum = 98.63%
Top op percent of field grouped dry gas = 98.63%
Top op percent of field grouped gross gas sum = 98.63%
Top op percent of field grouped oil and condensate sum = 98.94%
Top op percent of field grouped ngl sum = 98.63%


In [ ]:
# select field or formation
row_choice = 'formation' # User can change this to 'formation'

if row_choice == 'formation':
    choice = ASKY_Form_ID_noNull
    name_col = 'formation_name'
    project_complex_col = 'formation_name'
elif row_choice == 'field':
    choice = ASKY_Field_ID_noNull
    name_col = 'field_name'
    project_complex_col = 'formation_name'
else:
    raise ValueError("Invalid row_choice. Must be 'formation' or 'field'.")

print(f'Choice is {row_choice}')
choice.head()

Choice is formation


,formation_name,operator_name,well_no,longitude,latitude,gross_gas_vol,dry_gas_vol,oil_and_condensate_vol,ngl_vol,resource_type,total_hydrocarbons_production,Entity ID
11,7 Rvrs-Q-Grayburg,Hilcorp Energy Company,30-025-22073; 30-025-27255; 30-025-11031; 30-0...,-103.192971,32.296161,0.237692,0.215727,0.007612,0.000497,Conventional,0.009378,E100001015870
26,7 Rvrs-Q-Grayburg,Southwest Royalties Inc,30-025-31794; 30-025-32107; 30-025-31539; 30-0...,-103.160461,32.344610,1.132815,1.028134,0.007835,0.002370,Conventional,0.016254,E100002021538
35,7 Rvrs-Qu-Gb-Sa,Mewbourne Oil Co,30-015-36619,-104.014305,32.701298,0.005069,0.004600,0.000002,0.000011,Conventional,0.000040,E100000191881
40,7Rvs-Qn-Gb-Glorieta-Yeso,Spur Energy Partners Llc,30-015-36309; 30-015-26754; 30-015-28518; 30-0...,-104.076770,32.820304,11.196198,10.161575,0.029035,0.023427,Conventional,0.112243,E100000192215
42,Abo,Apache Corporation,30-015-01554; 30-015-01534; 30-015-03194; 30-0...,-104.178889,32.773297,6.845683,6.213084,0.033087,0.014324,Conventional; Unconventional,0.083963,E100001014128


In [ ]:
# standardize the 'resource_type' column for GOGET bulk upload

choice['resource_type'] = choice['resource_type'].replace({
    'Unconventional; Conventional': 'Unconventional',
    'Conventional; Unconventional': 'Unconventional',
    'Conventional': 'Conventional'
})

print("Resource types standardized.")

Resource types standardized.


In [ ]:
# define fuel type
import numpy as np

conditions = [
    (choice['dry_gas_vol'] > 0) & (choice['oil_and_condensate_vol'] > 0),
    (choice['dry_gas_vol'] == 0) & (choice['oil_and_condensate_vol'] > 0),
    (choice['dry_gas_vol'] > 0) & (choice['oil_and_condensate_vol'] == 0)
]

choices = ['oil and gas', 'oil', 'gas']

choice['fuel_type'] = np.select(conditions, choices, default=None)

print("Fuel types have been assigned.")

Fuel types have been assigned.


In [ ]:
# format names based on field or formation list - add source
choice.loc[:, 'name'] = choice[name_col] + ' - ' + choice['operator_name']
choice.loc[:, 'unit_type'] = "asset"
choice.loc[:, 'project_complex'] = choice[project_complex_col]
choice.loc[:, 'coordinate_accuracy'] = "exact"
choice.loc[:, 'operator_source'] = "OngardNMProd2024"
choice.loc[:, 'location_source'] = "OngardNMProd2024"
choice.loc[:, 'resource_type_source'] = "OngardNMProd2024"
choice.loc[:, 'discovery_year_source'] = "OngardNMProd2024"
choice['name'] = choice['name'].str.replace('Wc', 'Wildcat', regex=False)

print("Data formatted for GOGET bulk upload.")

Data formatted for GOGET bulk upload.


In [ ]:
# add total production column if it doesn't already exist (it should from the earlier step, but this is a safeguard)
if 'total_hydrocarbons_production' not in choice.columns:
    choice['total_hydrocarbons_production'] = (choice['dry_gas_vol']*0.005883) + choice['oil_and_condensate_vol'] + choice['ngl_vol']

# Select and order the final columns for ASKY_Clean
ASKY_Clean = choice[[
    'name',
    'unit_type',
    'fuel_type',
    'project_complex',
    'longitude',
    'latitude',
    'coordinate_accuracy',
    'location_source',
    'operator_name',
    'operator_source',
    'Entity ID',
    'resource_type',
    'resource_type_source',
    # 'Discovery year',
    # 'discovery_year_source',
    'well_no',
    'total_hydrocarbons_production'
]].copy()

# sort by total_hydrocarbons_production high to low
ASKY_Clean = ASKY_Clean.sort_values(by='total_hydrocarbons_production', ascending=False).reset_index(drop=True)

print("ASKY_Clean DataFrame created with selected and ordered columns.")
display(ASKY_Clean.head())

ASKY_Clean DataFrame created with selected and ordered columns.


,name,unit_type,fuel_type,project_complex,longitude,latitude,coordinate_accuracy,location_source,operator_name,operator_source,Entity ID,resource_type,resource_type_source,well_no,total_hydrocarbons_production
0,Formation not assigned - Eog Resources Inc,asset,oil and gas,Formation not assigned,-103.620202,32.169670,exact,OngardNMProd2024,Eog Resources Inc,OngardNMProd2024,E100001014954,Unconventional,OngardNMProd2024,30-025-52244; 30-025-52256; 30-025-51708; 30-0...,234.996064
1,Formation not assigned - Devon Energy Producti...,asset,oil and gas,Formation not assigned,-103.728304,32.278582,exact,OngardNMProd2024,"Devon Energy Production Company, Lp",OngardNMProd2024,E100000191945,Unconventional,OngardNMProd2024,30-015-49343; 30-015-49344; 30-015-49349; 30-0...,145.724158
2,Formation not assigned - Mewbourne Oil Co,asset,oil and gas,Formation not assigned,-103.937203,32.362817,exact,OngardNMProd2024,Mewbourne Oil Co,OngardNMProd2024,E100000191881,Unconventional,OngardNMProd2024,30-025-49534; 30-015-53958; 30-015-54090; 30-0...,145.496052
3,Formation not assigned - Oxy Usa Inc,asset,oil and gas,Formation not assigned,-103.789442,32.843511,exact,OngardNMProd2024,Oxy Usa Inc,OngardNMProd2024,E100002015587,Unconventional,OngardNMProd2024,30-025-50307; 30-025-50306; 30-025-52491; 30-0...,90.638675
4,Formation not assigned - Matador Production Co...,asset,oil and gas,Formation not assigned,-103.717206,32.338140,exact,OngardNMProd2024,Matador Production Company,OngardNMProd2024,E100000191956,Unconventional,OngardNMProd2024,30-025-49184; 30-025-47196; 30-015-53894; 30-0...,83.867639


In [ ]:
# check totals in choice
ASKY_choice_total = ASKY_Clean['total_hydrocarbons_production'].sum()


# print totals
print(f'{active_state} parsed data has {ASKY_choice_total} million boe')

new_mexico parsed data has 1473.88347745201 million boe


In [ ]:
# save out ASKY_Clean for manual matching
ASKY_Clean.to_excel(f'{NM_GOGET_Folder}/NM_GOGET_Clean_for_ID_Match.xlsx', index=False)